# Enhanced HK-RAG with Adaptive Filtering & Diagnostic Tools

This notebook implements improvements to the Hybrid Knowledge RAG system:
1. **Diagnostic Logging**: Track why hybrid mode underperforms vs history-only
2. **Adaptive Threshold**: Higher similarity threshold (0.65) for hybrid mode 
3. **Weighted Prompts**: Hierarchical structuring to prioritize patient history
4. **Experimental Framework**: Systematic comparison of strategies

In [ ]:
# ============================================================================
# CELL 1: CONFIGURATION & CONSTANTS
# ============================================================================

import re, json, time
import hashlib
import pickle
import os
import atexit
import asyncio
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime

import pandas as pd
import tiktoken
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import openai

# =========================
# Path Configuration
# =========================

TRANSCRIPT_ROOT = Path("Transcripts_Deepgram_nova-2_order by patient")
PREVREPORTS_DIR = Path("PrevReports")
PATIENT_PROFILE_CSV = Path("patient_profiles copy.csv")
FORM_ROOT = Path("data/Visiting Form/Visiting Form - Order by Patient")

OUT_DIR = Path("cross_patient_rag_output_deepgram_v1_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# API Configuration
# =========================

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# =========================
# Model Configuration
# =========================

MODEL_EXTRACT = "gpt-4o"           # Extraction: Complex parsing of medical conversations
MODEL_SUMMARY = "gpt-4o"           # Summary: Structured output, sufficient quality
TEMP_EXTRACT = 0.0                  # Deterministic extraction (factual data only)
TEMP_SUMMARY = 0.3                  # Consistent but natural medical writing

# =========================
# Field Configuration
# =========================

FIELDS = ["体温", "血圧", "脈拍", "呼吸", "SPO2", "排便", "食欲"]

# =========================
# Concurrency Settings
# =========================

MAX_CONCURRENCY = 6  # Rate limit guard

# =========================
# Official Form Template
# =========================

OFFICIAL_FORM_TEMPLATE = """
1.【利用者の状態（病状・トピックス）】
このセクションは以下の内容で構成されています。

- トピックス: 
  今回の診察で最も重要な臨床観察について記述してください。
判断基準：
  1. 安全リスク: 転倒、出血、強い痛み、発熱（38℃以上）、意識変容、呼吸困難
  2. 症状の変化: 新たな症状、既存症状の悪化・改善、処置の変更
  3. 日常生活への影響: ADL低下、活動制限、睡眠障害
  4. 変化なし: 上記が該当しない場合のみ「特に重大な問題なし」
  
  【記載方法】
  - 具体的に記載
  - 時系列を明記: 「いつ」「何が」「どう変化したか」を含める
  - 臨床的意義: 日常生活や今後のケアへの影響を示唆

- フォームから抽出した各カテゴリーを【..】内にリストアップしてください。各カテゴリごとに、観察した事実を**臨床的文脈**と共に説明します。
  【記載ルール】
  1. 単なる事実の羅列ではなく、臨床的に意味のある文章で記載
  2. 複数の観察を関連付けて記載（例: 睡眠と疼痛の関係）
  3. 時系列や因果関係があれば明記
  4. 患者の主観的訴えと客観的観察を区別

- <処置内容・訓練内容>: 会話に基づき、実施した処置や訓練の内容を記載。
- <備考>: 特別な注意事項や重要な補足情報を記載。
- <外部連絡・伝達事項>: 外部機関（医師、ケアマネージャー、家族など）への連絡事項を記載。
- <内部連絡・伝達事項>: 事業所内のスタッフ間での申し送り事項を記載。

2.【実施した看護・リハビリテーションの内容】
※会話から推測できる実施内容を記載。生活援助、排泄援助、処置、リハビリ、指導、管理の6点をすべて書きます。何も一致しない場合は空白のままにしてください。
    - 生活援助: [具体的な実施内容]（例：バイタルチェック、状態観察、全身清拭、ひげそり、爪切り、水分摂取介助、精神的援助、介護者支援）
    - 排泄援助: [具体的な実施内容]（例：パウチ交換、オムツ交換、パット交換）
    - 処置: [具体的な実施内容]（例：外用薬塗布、創処置、褥瘡処置、キャビロン塗布、パッチ貼付）
    - リハビリ: [具体的な実施内容]（例：呼吸訓練、歩行訓練）
    - 指導: [具体的な実施内容]（例：観察方法指導、合併症予防指導、排便調整指導、食事指導、服薬指導、社会資源活用指導、緊急時対応指導、在宅酸素療法指導、介護指導、ＡＤＬ指導、療養指導、水分摂取指導）
    - 管理: [具体的な実施内容]（例：ストマ管理、薬物管理、在宅酸素管理）

3.【コメント】
会話に基づく今後の予定や受診予定を記載（施設名・日付など）。
""".strip()

print("✅ Configuration loaded successfully!")
print(f"📊 Extraction Model: {MODEL_EXTRACT} (temp={TEMP_EXTRACT})")
print(f"📝 Summary Model: {MODEL_SUMMARY} (temp={TEMP_SUMMARY})")
print(f"🔧 Max Concurrency: {MAX_CONCURRENCY}")
print(f"📁 Output Directory: {OUT_DIR}")

In [ ]:
# ============================================================================
# CELL 2: CACHE MANAGEMENT & METERING
# ============================================================================

# =========================
# Cache Setup
# =========================

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
EXTRACTION_CACHE_FILE = CACHE_DIR / "extraction_cache.pkl"
EMBEDDING_CACHE_FILE = CACHE_DIR / "embedding_cache.pkl"
KNOWLEDGE_CACHE_FILE = CACHE_DIR / "knowledge_cache.pkl"

# Initialize global caches
EXTRACTION_CACHE = {}
EMBEDDING_CACHE = {}
KNOWLEDGE_CACHE = {}

def load_cache(cache_file: Path) -> dict:
    """Load cache from disk if it exists"""
    if cache_file.exists():
        try:
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Warning: Could not load cache {cache_file}: {e}")
    return {}

def save_cache(cache_obj: dict, cache_file: Path) -> None:
    """Safe cache saving with atomic write"""
    try:
        tmp_path = cache_file.with_suffix(cache_file.suffix + ".tmp")
        with open(tmp_path, 'wb') as f:
            pickle.dump(cache_obj, f)
        os.replace(tmp_path, cache_file)
    except Exception as e:
        print(f"Warning: Could not save cache {cache_file}: {e}")

def persist_all_caches():
    """Persist all caches at exit"""
    save_cache(EXTRACTION_CACHE, EXTRACTION_CACHE_FILE)
    save_cache(EMBEDDING_CACHE, EMBEDDING_CACHE_FILE)
    save_cache(KNOWLEDGE_CACHE, KNOWLEDGE_CACHE_FILE)
    print("💾 Caches persisted to disk.")

# Register cleanup handler
atexit.register(persist_all_caches)

# Load existing caches
EXTRACTION_CACHE = load_cache(EXTRACTION_CACHE_FILE)
EMBEDDING_CACHE = load_cache(EMBEDDING_CACHE_FILE)
KNOWLEDGE_CACHE = load_cache(KNOWLEDGE_CACHE_FILE)

# =========================
# Metering System
# =========================

class ProcessingMeter:
    """Track time, API calls, and costs for batch processing"""
    
    def __init__(self):
        self.start_time = None
        self.end_time = None
        self.api_calls = {
            'extraction': 0,
            'embedding': 0, 
            'summary': 0,
            'total': 0
        }
        self.tokens_used = {
            'input_tokens': 0,
            'output_tokens': 0,
            'total_tokens': 0
        }
        self.costs = {
            'extraction_cost': 0.0,
            'embedding_cost': 0.0,
            'summary_cost': 0.0,
            'total_cost': 0.0
        }
        
        # OpenAI pricing (as of 2024)
        self.pricing = {
            'gpt-4o': {'input': 2.50, 'output': 10.00},  # per 1M tokens
            'gpt-4o-mini': {'input': 0.15, 'output': 0.60},  # per 1M tokens
            'text-embedding-3-small': 0.02  # per 1M tokens
        }
    
    def start_timer(self):
        """Start timing the process"""
        self.start_time = time.time()
        print(f"🕐 Processing started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    def end_timer(self):
        """End timing and calculate duration"""
        self.end_time = time.time()
        duration = self.end_time - self.start_time
        print(f"⏰ Processing completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⌛ Total duration: {self.format_duration(duration)}")
        return duration
    
    def format_duration(self, seconds):
        """Format duration in readable format"""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        return f"{hours}h {minutes}m {secs}s"
    
    def track_api_call(self, call_type, model, input_tokens=0, output_tokens=0):
        """Track an API call and calculate cost"""
        self.api_calls[call_type] += 1
        self.api_calls['total'] += 1
        
        # Track tokens
        self.tokens_used['input_tokens'] += input_tokens
        self.tokens_used['output_tokens'] += output_tokens
        self.tokens_used['total_tokens'] += input_tokens + output_tokens
        
        # Calculate cost
        cost = 0.0
        if model in self.pricing:
            if model == 'text-embedding-3-small':
                cost = (input_tokens / 1_000_000) * self.pricing[model]  # Per 1M tokens
            else:
                cost = (input_tokens / 1_000_000) * self.pricing[model]['input'] + \
                       (output_tokens / 1_000_000) * self.pricing[model]['output']  # Per 1M tokens
        
        # Add to appropriate cost category
        self.costs[f'{call_type}_cost'] += cost
        self.costs['total_cost'] += cost
        
        return cost
    
    def print_summary(self):
        """Print detailed summary of processing metrics"""
        duration = self.end_time - self.start_time if self.end_time else 0
        
        print(f"\n{'='*60}")
        print(f"📊 PROCESSING SUMMARY")
        print(f"{'='*60}")
        
        print(f"⏱️  TIMING:")
        print(f"   • Duration: {self.format_duration(duration)}")
        print(f"   • Start: {datetime.fromtimestamp(self.start_time).strftime('%H:%M:%S')}")
        print(f"   • End: {datetime.fromtimestamp(self.end_time).strftime('%H:%M:%S')}")
        
        print(f"\n📞 API CALLS:")
        for call_type, count in self.api_calls.items():
            print(f"   • {call_type.title()}: {count}")
        
        print(f"\n🔤 TOKENS USED:")
        print(f"   • Input tokens: {self.tokens_used['input_tokens']:,}")
        print(f"   • Output tokens: {self.tokens_used['output_tokens']:,}")
        print(f"   • Total tokens: {self.tokens_used['total_tokens']:,}")
        
        print(f"\n💰 COSTS:")
        for cost_type, amount in self.costs.items():
            print(f"   • {cost_type.replace('_', ' ').title()}: ${amount:.4f}")
        
        print(f"\n📈 EFFICIENCY:")
        if self.api_calls['total'] > 0:
            avg_time_per_call = duration / self.api_calls['total']
            avg_cost_per_call = self.costs['total_cost'] / self.api_calls['total']
            print(f"   • Average time per API call: {avg_time_per_call:.2f}s")
            print(f"   • Average cost per API call: ${avg_cost_per_call:.4f}")
        
        print(f"{'='*60}")

# Global meter instance
processing_meter = ProcessingMeter()

print("✅ Cache management initialized!")
print(f"📦 Loaded {len(EXTRACTION_CACHE)} extraction cache entries")
print(f"📦 Loaded {len(EMBEDDING_CACHE)} embedding cache entries")
print(f"📦 Loaded {len(KNOWLEDGE_CACHE)} knowledge cache entries")
print("✅ Processing meter ready!")

In [ ]:
# ============================================================================
# CELL 3: EXTRACTION FUNCTIONS
# ============================================================================

# =========================
# OpenAI Client Helper
# =========================

async def get_optimized_openai_client():
    """Get optimized OpenAI client"""
    return openai.AsyncOpenAI(api_key=OPENAI_API_KEY, timeout=60.0, max_retries=3)

# =========================
# Extraction Prompt
# =========================

STAGE_A_PROMPT = """あなたは、在宅看護の会話記録から健康情報を抽出するAIアシスタントです。
会話で明示的に言及されていない情報は、一切出力しないでください。

# 出力要件
- 会話内容に明示的な値があれば：必ずその値を最優先で使用してください。
- 出力は有効なJSONのみ（コードブロック・説明文・末尾カンマなし）。
- JSONキーは次の7項目のみ：体温, 血圧, 脈拍, 呼吸, SPO2, 排便, 食欲。
- 会話に値が存在しない場合、`"value"`は`null`、`"evidence"`は空文字(`""`)としてください。

【数値項目（体温・血圧・脈拍・SPO2）の判断ルール】
1. 信頼できる文脈でのみ抽出する: 値が「測定した」「計った」「です」「でした」などの動詞や断定の表現と共に言及されている場合を最優先する。
2. 明らかに異常な値は却下する: 体温が34°C未満または42°C以上、血圧の舒张期（下の値）が收缩期（上の値）より高い、SPO2が70%未満などの医学的に不可能・`null`とする。
3. 単独の数値に注意:
    - 血圧は「120/80」のような２つの数値の組でない場合は`null`とする。「今日が150」のような発言は収縮期のみの可能性が高く、抽出しない。
4. その他会話中の数値を無視する: 日付（5月7日）、時間、年代、その他明らかに vital sign に関係ない数字は完全に無視する。

【排便・食欲について】
- 排便・食欲は、「有/無」を判断できる明確な発言（例：「便があった」「食べた」「食べていない」）。
- 「有/無」で出力するのは、会話で明確に言及されている場合のみです。

【排便の判断ルール】
1. 「有」とするのは「本日」の排便を直接示す発言のみ: 「出た」「あった」「便通があった」「下痢をした」
2.「無」とするのは「本日」の排便が無いと直接示す発言のみ: 「出ていない」「ない」「まだ」「便秘」
3. 以下の場合は「null」とする:
    - 「昨日出た」など、過去の分のみの言及。
    - 「便を捨ててない」など、処理に関する発言。
    - 「お通じは？」への返答がない、または「少し」「普通」などの曖昧な表現。

【食欲の判断ルール】
食欲の判断ルール（改善版）:
1. 「有（良好）」: 「全部食べた」「完食」「普通に食べた」
2. 「有（不良）」: 「少し食べた」「半分食べた」「あまり食べていない」
3. 「無」: 「食べていない」「全く食べていない」
4. 「null」: 明示的言及なし

#【出力形式例】
{{
  "体温":   {{"value": "36.4°C", "evidence": "熱は三十六度四分"}},
  "血圧":   {{"value": "124/60 mmHg", "evidence": "血圧が124の60"}},
  "脈拍":   {{"value": null, "evidence": ""}},
  "呼吸":   {{"value": null, "evidence": ""}},
  "SPO2":  {{"value": null, "evidence": ""}},
  "排便":   {{"value": "無", "evidence": "この2日間は出ていない"}},
  "食欲":   {{"value": "有", "evidence": "ご飯は全部食べました"}}
}}

【会話内容】
{conversation}
""".strip()

# =========================
# Extraction Function
# =========================

async def extract_with_json_mode(transcript: str, patient_id: str) -> Dict[str, Any]:
    """Optimized Stage A extraction with enforced JSON mode and caching - WITH METERING"""
    
    # Check cache first
    cache_key = hashlib.sha256(transcript.encode()).hexdigest()
    if cache_key in EXTRACTION_CACHE:
        print(f"🎯 Cache HIT for extraction: {patient_id}")
        return EXTRACTION_CACHE[cache_key]
    
    try:
        client = await get_optimized_openai_client()
        
        # Use the detailed STAGE_A_PROMPT
        prompt_content = STAGE_A_PROMPT.format(conversation=transcript)
        
        # Count input tokens for metering
        try:
            encoding = tiktoken.encoding_for_model(MODEL_EXTRACT)
        except Exception:
            encoding = tiktoken.get_encoding("o200k_base")
        input_tokens = len(encoding.encode(prompt_content))
        
        response = await client.chat.completions.create(
            model=MODEL_EXTRACT,
            messages=[{"role": "user", "content": prompt_content}],
            temperature=TEMP_EXTRACT,
            response_format={"type": "json_object"}
        )
        
        # Count output tokens and track API call
        output_tokens = response.usage.completion_tokens if response.usage else 0
        actual_input_tokens = response.usage.prompt_tokens if response.usage else input_tokens
        
        # Track this API call in the meter
        processing_meter.track_api_call('extraction', MODEL_EXTRACT, actual_input_tokens, output_tokens)
        
        # Parse JSON safely
        try:
            result = json.loads(response.choices[0].message.content)
        except Exception:
            result = {field: {"value": None, "evidence": ""} for field in FIELDS}
        
        # Cache the result
        EXTRACTION_CACHE[cache_key] = result
        # Periodically save cache
        if len(EXTRACTION_CACHE) % 10 == 0:
            save_cache(EXTRACTION_CACHE, EXTRACTION_CACHE_FILE)
            
        print(f"✅ JSON extraction successful for {patient_id}")
        return result
        
    except Exception as e:
        print(f"❌ JSON extraction failed for {patient_id}: {e}")
        # Fallback to empty structure
        return {field: {"value": None, "evidence": ""} for field in FIELDS}

# =========================
# Save Extraction Results
# =========================

async def save_extraction_results(extraction_result: Dict[str, Any], patient_id: str, transcript_file: str = "") -> str:
    """Save extraction results to JSON file with filename matching transcript pattern"""
    
    # Create extraction results directory
    extraction_dir = OUT_DIR / "extraction_results"
    extraction_dir.mkdir(exist_ok=True)
    
    # Generate filename based on transcript file pattern (no timestamp needed)
    if transcript_file:
        transcript_path = Path(transcript_file)
        base_name = transcript_path.stem  # e.g., "audio_example_visit_0001"
        filename = f"extraction_{patient_id}_{base_name}.json"
    else:
        # Fallback: use timestamp only if no transcript file provided
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"extraction_{patient_id}_{timestamp}.json"
    
    output_file = extraction_dir / filename
    
    # Prepare complete results with metadata
    complete_result = {
        "patient_id": patient_id,
        "extraction_timestamp": datetime.now().isoformat(),
        "transcript_file": transcript_file,
        "extraction_data": extraction_result,
        "fields_extracted": list(extraction_result.keys()) if isinstance(extraction_result, dict) else [],
        "extraction_method": "stage_a_prompt_json_mode"
    }
    
    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(complete_result, f, ensure_ascii=False, indent=2)
    
    print(f"💾 Extraction results saved to: {output_file}")
    return str(output_file)

print("✅ Extraction functions loaded!")
print("📋 Functions available:")
print("   • extract_with_json_mode() - JSON-mode extraction with caching")
print("   • save_extraction_results() - Save results to structured JSON")
print("🎯 Features: Medical validation, caching, cost tracking")

In [ ]:
# ============================================================================
# CELL 4: EMBEDDING FUNCTIONS
# ============================================================================

# =========================
# Text Chunking
# =========================

def chunk_text_for_embedding(text: str, max_tokens: int = 8000) -> List[str]:
    """
    Split text into chunks that fit within embedding model token limits
    
    Args:
        text: Input text to chunk
        max_tokens: Maximum tokens per chunk (default 8000 to stay under 8192 limit)
        
    Returns:
        List of text chunks
    """
    if not text or text.strip() == "":
        return [""]
    
    try:
        # Get tokenizer
        try:
            encoding = tiktoken.encoding_for_model("text-embedding-3-small")
        except Exception:
            encoding = tiktoken.get_encoding("cl100k_base")  # Fallback
        
        # Check if text fits within limit
        tokens = encoding.encode(text)
        if len(tokens) <= max_tokens:
            return [text]
        
        print(f"⚠️ Text too long ({len(tokens)} tokens), chunking into smaller pieces...")
        
        # Split by sentences first (better semantic boundaries)
        sentences = text.split('。')  # Japanese sentence delimiter
        if len(sentences) == 1:
            sentences = text.split('.')  # English fallback
        if len(sentences) == 1:
            sentences = text.split('\n')  # Line breaks fallback
        
        chunks = []
        current_chunk = ""
        current_tokens = 0
        
        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
                
            sentence_tokens = len(encoding.encode(sentence + '。'))
            
            # If single sentence is too long, split it further
            if sentence_tokens > max_tokens:
                # Force split by character count as last resort
                char_limit = int(max_tokens * 3)  # Rough estimate: 1 token ≈ 3-4 chars
                for i in range(0, len(sentence), char_limit):
                    chunk_part = sentence[i:i + char_limit]
                    if chunk_part.strip():
                        chunks.append(chunk_part.strip())
                continue
            
            # Check if adding this sentence would exceed limit
            if current_tokens + sentence_tokens > max_tokens and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence + '。'
                current_tokens = sentence_tokens
            else:
                current_chunk += sentence + '。'
                current_tokens += sentence_tokens
        
        # Add remaining chunk
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        
        print(f"✅ Split into {len(chunks)} chunks")
        return chunks
        
    except Exception as e:
        print(f"Error in chunking: {e}")
        # Emergency fallback: split by character count
        char_limit = 24000  # Very rough estimate for 8000 tokens
        chunks = []
        for i in range(0, len(text), char_limit):
            chunk = text[i:i + char_limit]
            if chunk.strip():
                chunks.append(chunk.strip())
        return chunks if chunks else [text]

# =========================
# Embedding Functions
# =========================

async def get_batch_embeddings(texts: List[str], embedding_model: str = "text-embedding-3-small") -> List[np.ndarray]:
    """
    Get embeddings for multiple texts in a single batch API call (more efficient)
    
    Args:
        texts: List of input texts
        embedding_model: OpenAI embedding model to use
        
    Returns:
        List of embedding vectors
    """
    if not texts:
        return []
    
    # Filter out empty texts and track their positions
    valid_texts = []
    valid_indices = []
    for i, text in enumerate(texts):
        if text and text.strip():
            valid_texts.append(text.strip())
            valid_indices.append(i)
    
    if not valid_texts:
        return [np.zeros(1536) for _ in texts]
    
    try:
        client = await get_optimized_openai_client()
        
        # ✅ BATCH PROCESSING - Send all texts at once
        response = await client.embeddings.create(
            input=valid_texts,
            model=embedding_model
        )
        
        # Track embedding API call with total tokens
        input_tokens = response.usage.total_tokens if response.usage else sum(len(t.split()) for t in valid_texts)
        processing_meter.track_api_call('embedding', embedding_model, input_tokens, 0)
        
        # Extract embeddings in order
        valid_embeddings = [np.array(data.embedding) for data in response.data]
        
        # Reconstruct full list with zeros for empty texts
        result = []
        valid_idx = 0
        for i in range(len(texts)):
            if i in valid_indices:
                result.append(valid_embeddings[valid_idx])
                valid_idx += 1
            else:
                result.append(np.zeros(1536))
        
        print(f"✅ Batch processed {len(valid_texts)} embeddings in 1 API call")
        return result
        
    except Exception as e:
        print(f"❌ Batch embedding failed: {e}, falling back to individual processing")
        # Fallback to individual processing
        return [await get_single_embedding(text, embedding_model) for text in texts]

async def get_single_embedding(text: str, embedding_model: str = "text-embedding-3-small") -> np.ndarray:
    """
    Get embedding for a single text (used as fallback or for single requests)
    
    Args:
        text: Input text
        embedding_model: OpenAI embedding model to use
        
    Returns:
        Embedding vector
    """
    if not text or text.strip() == "":
        return np.zeros(1536)
    
    try:
        client = await get_optimized_openai_client()
        response = await client.embeddings.create(
            input=text.strip(),
            model=embedding_model
        )
        
        input_tokens = response.usage.total_tokens if response.usage else len(text.split())
        processing_meter.track_api_call('embedding', embedding_model, input_tokens, 0)
        
        return np.array(response.data[0].embedding)
        
    except Exception as e:
        print(f"❌ Single embedding failed: {e}")
        return np.zeros(1536)

async def get_averaged_embedding(text: str, embedding_model: str = "text-embedding-3-small") -> np.ndarray:
    """
    Get embedding for potentially long text by chunking and averaging with BATCH PROCESSING
    
    Args:
        text: Input text (can be longer than model limit)
        embedding_model: OpenAI embedding model to use
        
    Returns:
        Averaged embedding vector
    """
    if not text or text.strip() == "":
        return np.zeros(1536)  # Standard embedding dimension
    
    try:
        # Get chunks that fit within model limits
        chunks = chunk_text_for_embedding(text)
        
        if len(chunks) == 1:
            # Single chunk, process normally
            return await get_single_embedding(chunks[0], embedding_model)
        
        else:
            # ✅ BATCH PROCESSING - Process all chunks in one API call
            print(f"🔄 Processing {len(chunks)} chunks for embedding with BATCH mode...")
            embeddings = await get_batch_embeddings(chunks, embedding_model)
            
            # Filter out any zero embeddings (failed chunks)
            valid_embeddings = [emb for emb in embeddings if not np.allclose(emb, 0)]
            
            if not valid_embeddings:
                print("❌ All chunks failed to embed")
                return np.zeros(1536)
            
            # Average the embeddings
            averaged_embedding = np.mean(valid_embeddings, axis=0)
            print(f"✅ Averaged {len(valid_embeddings)} chunk embeddings using BATCH processing")
            return averaged_embedding
            
    except Exception as e:
        print(f"❌ Error in get_averaged_embedding: {e}")
        return np.zeros(1536)

print("✅ Embedding functions loaded!")
print("📋 Functions available:")
print("   • chunk_text_for_embedding() - Smart text chunking")
print("   • get_batch_embeddings() - Batch process multiple texts (10x faster!)")
print("   • get_single_embedding() - Single text embedding")
print("   • get_averaged_embedding() - Handle long texts with chunking")
print("⚡ Batch processing: Up to 10x faster and more cost-effective!")

In [ ]:
# ============================================================================
# CELL 5: RAG SYSTEM CLASSES
# ============================================================================

# =========================
# Helper Functions
# =========================

def read_transcript_text(transcript_path: Path) -> str:
    """Read transcript text from file"""
    try:
        with open(transcript_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading transcript {transcript_path}: {e}")
        return ""

def load_patient_history(patient_id: str, prevreports_dir: Path) -> List[Dict]:
    """Load patient history from previous reports"""
    history = []
    
    # subfolder structure: PrevReports/P*/PrevReports - P*.csv
    patient_subfolder = prevreports_dir / patient_id
    if patient_subfolder.exists():
        prev_file = patient_subfolder / f"PrevReports - {patient_id}.csv"
        if prev_file.exists():
            try:
                df = pd.read_csv(prev_file)
                if not df.empty and 'Field' in df.columns and 'Value' in df.columns:
                    field_value_dict = dict(zip(df['Field'], df['Value']))
                    transcription = field_value_dict.get('Transcription', '')
                    date = field_value_dict.get('Date', 'Unknown')
                    
                    if transcription and str(transcription).strip():
                        history.append({
                            'text': str(transcription).strip(),
                            'date': str(date)
                        })
                        print(f"✅ Loaded patient history for {patient_id}: {len(str(transcription))} chars")
            except Exception as e:
                print(f"❌ Error loading history for {patient_id}: {e}")
    
    return history

async def load_all_patient_data():
    """Load all patient data for cross-patient knowledge base"""
    all_patient_data: Dict[str, Dict[str, Any]] = {}
    
    # Helper to find a column by candidate names
    def find_col(cols: List[str], candidates: List[str]) -> Optional[str]:
        lower_map = {c.lower(): c for c in cols}
        for name in candidates:
            if name.lower() in lower_map:
                return lower_map[name.lower()]
        return None
    
    # Load patient profiles
    if PATIENT_PROFILE_CSV.exists():
        try:
            profiles_df = pd.read_csv(PATIENT_PROFILE_CSV)
            for _, row in profiles_df.iterrows():
                cols = list(row.index)
                pid_col = find_col(cols, ['patient_id'])
                disease_col = find_col(cols, ['disease'])
                gender_col = find_col(cols, ['gender'])
                age_col = find_col(cols, ['age'])
                
                patient_id = str(row.get(pid_col) if pid_col else row.get('Patient_ID', 'Unknown'))
                
                # Build structured profile dict
                raw_gender = str(row.get(gender_col, '') if gender_col else '').strip()
                g = raw_gender.lower()
                if g in ['m', 'M', '男性']:
                    gender_std = 'male'
                elif g in ['f', 'F', '女性']:
                    gender_std = 'female'
                else:
                    gender_std = raw_gender.lower() if raw_gender else None
                
                age_val = row.get(age_col) if age_col else None
                try:
                    age_std = int(age_val) if pd.notna(age_val) else None
                except Exception:
                    age_std = None
                
                profile_struct = {
                    'disease': str(row.get(disease_col, '')) if disease_col else '',
                    'gender': gender_std,
                    'age': age_std,
                }
                
                all_patient_data[patient_id] = {
                    'profile': profile_struct,
                    'history': []
                }
        except Exception as e:
            print(f"Error loading patient profiles: {e}")
    
    # Load previous reports for each patient
    for patient_id in list(all_patient_data.keys()):
        history = load_patient_history(patient_id, PREVREPORTS_DIR)
        all_patient_data[patient_id]['history'] = history
    
    print(f"📊 Loaded data for {len(all_patient_data)} patients")
    return all_patient_data

# =========================
# Strategy Selector (ENHANCED with Adaptive Threshold)
# =========================

class AdaptiveRAGStrategy:
    """Hybrid strategy with adaptive threshold - Maximize knowledge utilization while maintaining safety"""
    
    def __init__(self, use_adaptive_threshold: bool = True):
        """
        Args:
            use_adaptive_threshold: If True, use higher similarity threshold for hybrid mode (0.65)
                                   If False, use original threshold (0.5)
        """
        self.use_adaptive_threshold = use_adaptive_threshold
    
    def select_strategy(self, patient_history_count, similar_patients_available):
        """Hybrid strategy selection - Use ALL available knowledge sources"""
        
        has_history = patient_history_count > 0
        has_similar = len(similar_patients_available) > 0
        
        if has_history and has_similar:
            return 'hybrid', f"✨ HYBRID: Using {patient_history_count} history + {len(similar_patients_available)} same-disease patients"
        elif has_history:
            return 'history_only', f"📝 HISTORY: Using {patient_history_count} previous reports only"
        elif has_similar:
            return 'cross_patient_only', f"🤝 CROSS-PATIENT: Using {len(similar_patients_available)} same-disease patients only"
        else:
            return 'no_rag', f"📄 TRANSCRIPT: No external knowledge available"
    
    def get_relevance_threshold(self, strategy: str) -> float:
        """
        Get adaptive semantic similarity threshold based on strategy
        
        Scientific Hypothesis:
        - In hybrid mode (patient history + cross-patient), cross-patient contexts need 
          higher quality bar to avoid noise when primary patient data exists
        - In cross-patient only mode, be more lenient as it's the only knowledge source
        
        Returns:
            float: Relevance threshold (0.5, 0.55, or 0.75)
        """
        if not self.use_adaptive_threshold:
            return 0.5  # Original threshold
        
        if strategy == 'hybrid':
            # STRICTER FILTERING: Only very high-quality cross-patient knowledge (increased from 0.65)
            # Rationale: Data shows cross-patient-only hurts performance, so only use when excellent
            return 0.75
        elif strategy == 'cross_patient_only':
            # More lenient when it's the only knowledge source
            return 0.55
        else:
            return 0.5
    
    def get_max_cross_patient_contexts(self, has_patient_history: bool) -> int:
        """
        Get maximum number of cross-patient contexts based on knowledge availability
        
        Scientific Rationale:
        - When patient history exists, limit cross-patient contexts to avoid noise
        - Volume imbalance (many cross-patient + few history) can mislead LLM
        - Only use top 2 highest-quality cross-patient matches in hybrid mode
        
        Args:
            has_patient_history: Whether patient-specific history is available
        
        Returns:
            int: Maximum number of cross-patient contexts to use
        """
        if has_patient_history:
            return 2  # Strict limit when history available (hybrid mode)
        else:
            return 5  # Allow more when it's the only knowledge source


# =========================
# Diagnostic Logger
# =========================

class HybridPerformanceDiagnostics:
    """
    Diagnostic logging to understand WHY hybrid underperforms vs history-only
    
    Tracks:
    - Retrieved cross-patient contexts and their similarity scores
    - Which contexts passed/failed relevance threshold
    - Systematic analysis of what makes hybrid fail
    """
    
    def __init__(self, output_file: Optional[Path] = None):
        self.logs = []
        self.output_file = output_file or Path("hybrid_diagnostics.json")
    
    def log_retrieval(self, 
                     patient_id: str,
                     strategy: str,
                     similar_patients: List[Dict],
                     retrieved_contexts: List[Dict],
                     relevance_threshold: float):
        """Log cross-patient retrieval details"""
        log_entry = {
            'patient_id': patient_id,
            'strategy': strategy,
            'relevance_threshold': relevance_threshold,
            'similar_patients_count': len(similar_patients),
            'similar_patients': [
                {
                    'patient_id': p['patient_id'],
                    'similarity_score': p['similarity_score'],
                    'disease': p.get('same_disease', 'Unknown')
                }
                for p in similar_patients
            ],
            'retrieved_contexts_count': len(retrieved_contexts),
            'retrieved_contexts': [
                {
                    'source_patient': ctx['source_patient'],
                    'relevance_score': ctx['relevance_score'],
                    'passed_threshold': ctx['relevance_score'] >= relevance_threshold,
                    'context_preview': ctx['context_text'][:150]
                }
                for ctx in retrieved_contexts
            ],
            'low_relevance_contexts': [
                ctx for ctx in retrieved_contexts 
                if ctx['relevance_score'] < 0.6
            ]
        }
        self.logs.append(log_entry)
    
    def analyze_failure_patterns(self, results_df: pd.DataFrame) -> Dict:
        """
        Analyze patterns in cases where history_only > hybrid
        
        Args:
            results_df: DataFrame with columns: patient_id, strategy, f1_score, etc.
        
        Returns:
            Analysis summary dict
        """
        # Merge to compare strategies
        hybrid_results = results_df[results_df['strategy'] == 'hybrid'].set_index('patient_id')
        history_results = results_df[results_df['strategy'] == 'history_only'].set_index('patient_id')
        
        common_patients = hybrid_results.index.intersection(history_results.index)
        
        if len(common_patients) == 0:
            return {'error': 'No overlapping patients between hybrid and history_only'}
        
        # Find cases where history_only outperforms hybrid
        failures = []
        for patient_id in common_patients:
            hybrid_f1 = hybrid_results.loc[patient_id, 'f1_score']
            history_f1 = history_results.loc[patient_id, 'f1_score']
            
            if history_f1 > hybrid_f1:
                # Find corresponding diagnostic log
                patient_logs = [log for log in self.logs if log['patient_id'] == patient_id]
                
                failures.append({
                    'patient_id': patient_id,
                    'hybrid_f1': float(hybrid_f1),
                    'history_f1': float(history_f1),
                    'degradation': float(history_f1 - hybrid_f1),
                    'diagnostic_log': patient_logs[0] if patient_logs else None
                })
        
        # Aggregate patterns
        analysis = {
            'total_comparisons': len(common_patients),
            'hybrid_failures': len(failures),
            'failure_rate': len(failures) / len(common_patients) if common_patients else 0,
            'mean_degradation': np.mean([f['degradation'] for f in failures]) if failures else 0,
            'failure_cases': failures
        }
        
        # Analyze low-relevance context patterns
        if failures:
            low_relevance_counts = []
            for failure in failures:
                if failure['diagnostic_log']:
                    low_count = len(failure['diagnostic_log'].get('low_relevance_contexts', []))
                    low_relevance_counts.append(low_count)
            
            if low_relevance_counts:
                analysis['mean_low_relevance_contexts'] = np.mean(low_relevance_counts)
        
        return analysis
    
    def save_logs(self):
        """Save diagnostic logs to JSON file"""
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump({
                'logs': self.logs,
                'summary': {
                    'total_logs': len(self.logs),
                    'strategies': list(set(log['strategy'] for log in self.logs))
                }
            }, f, ensure_ascii=False, indent=2)
        print(f"📊 Diagnostic logs saved to: {self.output_file}")

# =========================
# Helper Functions
# =========================

def strip_numeric_data(text: str) -> str:
    """
    Remove numeric data from historical context to reduce tokens.
    
    Rationale:
    - Current transcript contains today's actual measurements
    - Historical context provides qualitative patterns, not specific numbers
    - Reduces token count by ~50% while preserving clinical relevance
    """
    if not text:
        return ""
    # Remove numbers (Arabic + full-width Japanese), percentages, dates
    text = re.sub(r"[0-9０-９％%/:\-\.]+", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

# =========================
# RAG System
# =========================

class SimplifiedCrossPatientRAGSystem:
    """Simplified Cross-Patient RAG System with metadata-enhanced embeddings"""
    
    def __init__(self, api_key):
        self.client = openai.AsyncOpenAI(api_key=api_key, timeout=60.0, max_retries=3)
        self.embedding_model = "text-embedding-3-small"
        self.patient_database = {}
        self.embedding_dim = 1536
        self.disease_index = {}
        
    def _build_metadata_prefix(self, metadata: Optional[Dict[str, Any]]) -> str:
        """Build structured metadata prefix for embedding enhancement"""
        if not metadata:
            return ""
        
        demographics = metadata.get('demographics', {})
        age = demographics.get('age')
        gender = demographics.get('gender')
        disease = demographics.get('primary_disease', '')
        
        # Convert gender to Japanese
        gender_jp = ''
        if gender == 'male' or gender == 'M':
            gender_jp = '男性'
        elif gender == 'female' or gender == 'F':
            gender_jp = '女性'
        
        # Build metadata sections
        metadata_parts = []
        if age is not None:
            metadata_parts.append(f"年齢: {age}歳")
        if gender_jp:
            metadata_parts.append(f"性別: {gender_jp}")
        if disease:
            metadata_parts.append(f"主な疾患: {disease}")
        
        if not metadata_parts:
            return ""
        
        prefix = "[患者情報: " + "、".join(metadata_parts) + "]\n\n"
        return prefix
    
    async def encode_text(self, text: str, metadata: Optional[Dict[str, Any]] = None) -> np.ndarray:
        """Encode text using OpenAI embeddings with optional metadata enhancement"""
        if not text or text.strip() == "":
            return np.zeros(self.embedding_dim)
        
        # Metadata enhancement
        enhanced_text = text
        if metadata:
            metadata_prefix = self._build_metadata_prefix(metadata)
            if metadata_prefix:
                enhanced_text = metadata_prefix + text
        
        cache_key = hashlib.sha256(enhanced_text.encode()).hexdigest()
        
        if cache_key in EMBEDDING_CACHE:
            return EMBEDDING_CACHE[cache_key]
        
        try:
            embedding = await get_averaged_embedding(enhanced_text, self.embedding_model)
            EMBEDDING_CACHE[cache_key] = embedding
            
            if len(EMBEDDING_CACHE) % 20 == 0:
                save_cache(EMBEDDING_CACHE, EMBEDDING_CACHE_FILE)
            
            return embedding
            
        except Exception as e:
            print(f"Embedding failed: {e}")
            return np.zeros(self.embedding_dim)
    
    async def build_cross_patient_knowledge_base(self, all_patient_data):
        """Build simplified knowledge base"""
        print("🏗️ Building Cross-Patient Knowledge Base...")
        
        self.patient_database = {}
        
        for patient_id, patient_info in all_patient_data.items():
            profile = await self.extract_simple_patient_profile(patient_id, patient_info)
            historical_contexts = await self.extract_historical_contexts(patient_id, patient_info)
            
            # Store metadata for each historical context
            for context in historical_contexts:
                context['patient_metadata'] = {
                    'demographics': profile['demographics']
                }
            
            self.patient_database[patient_id] = {
                'profile': profile,
                'history': historical_contexts,
                'raw_data': patient_info
            }
        
        print(f"✅ Knowledge Base built: {len(self.patient_database)} patients")
        
        # Create disease index for same-disease matching
        self.disease_index = {}
        for patient_id, patient_data in self.patient_database.items():
            primary_disease = patient_data['profile']['demographics'].get('primary_disease', '').lower().strip()
            if primary_disease:
                if primary_disease not in self.disease_index:
                    self.disease_index[primary_disease] = []
                self.disease_index[primary_disease].append(patient_id)
        
        print(f"🔒 Disease index created: {len(self.disease_index)} unique diseases")
    
    async def extract_simple_patient_profile(self, patient_id, patient_info):
        """Extract demographics only"""
        profile_input = patient_info.get('profile', {})
        demographics = self.extract_demographics(profile_input)
        
        return {
            'patient_id': patient_id,
            'demographics': demographics
        }
    
    def extract_demographics(self, profile_data):
        """Extract demographics from profile data"""
        demographics = {}
        
        if isinstance(profile_data, dict):
            age_val = profile_data.get('age')
            try:
                demographics['age'] = int(age_val) if age_val is not None else None
            except Exception:
                demographics['age'] = None
            
            gender_raw = str(profile_data.get('gender', '')).strip()
            if gender_raw:
                g = gender_raw.lower()
                if g in ['m', 'male', 'M', '男性']:
                    demographics['gender'] = 'male'
                elif g in ['f', 'female', 'F', '女性']:
                    demographics['gender'] = 'female'
                else:
                    demographics['gender'] = g
            
            disease_raw = str(profile_data.get('disease', '')).strip()
            if disease_raw:
                demographics['primary_disease'] = disease_raw
        
        return demographics
    
    async def extract_historical_contexts(self, patient_id, patient_info):
        """Extract historical contexts from patient data"""
        contexts = []
        history_data = patient_info.get('history', [])
        
        for record in history_data:
            if isinstance(record, dict):
                contexts.append({
                    'text': record.get('text', ''),
                    'date': record.get('date', 'Unknown'),
                    'type': 'historical_visit'
                })
            elif isinstance(record, str):
                contexts.append({
                    'text': record,
                    'date': 'Unknown',
                    'type': 'historical_visit'
                })
        
        return contexts
    
    def find_same_disease_patients_only(self, target_patient_id: str, top_k: int = 3) -> List[Dict]:
        """Find patients with EXACTLY the same disease only (safety-first)"""
        if target_patient_id not in self.patient_database:
            return []
        
        target_profile = self.patient_database[target_patient_id]['profile']
        target_disease = target_profile['demographics'].get('primary_disease', '').lower().strip()
        
        if not target_disease:
            return []
        
        same_disease_patients = self.disease_index.get(target_disease, [])
        same_disease_patients = [pid for pid in same_disease_patients if pid != target_patient_id]
        
        if not same_disease_patients:
            return []
        
        # Calculate simplified similarity
        similarities = []
        target_demographics = target_profile['demographics']
        
        for patient_id in same_disease_patients:
            if patient_id in self.patient_database:
                candidate_profile = self.patient_database[patient_id]['profile']
                similarity_score = self.calculate_simplified_similarity(
                    target_demographics, 
                    candidate_profile['demographics']
                )
                
                similarities.append({
                    'patient_id': patient_id,
                    'similarity_score': similarity_score,
                    'profile': candidate_profile,
                    'history': self.patient_database[patient_id]['history'],
                    'same_disease': target_disease
                })
        
        similarities.sort(key=lambda x: x['similarity_score'], reverse=True)
        return similarities[:top_k]
    
    def calculate_simplified_similarity(self, demographics_a, demographics_b):
        """Calculate basic demographic similarity (age + gender)"""
        total_score = 0.0
        
        # Age similarity (70% weight)
        age_a = demographics_a.get('age')
        age_b = demographics_b.get('age')
        
        if age_a is not None and age_b is not None:
            age_diff = abs(age_a - age_b)
            if age_diff <= 5:
                age_similarity = 1.0
            elif age_diff <= 10:
                age_similarity = 0.8
            elif age_diff <= 15:
                age_similarity = 0.6
            else:
                age_similarity = 0.4
        else:
            age_similarity = 0.5
        
        total_score += age_similarity * 0.7
        
        # Gender similarity (30% weight)
        gender_a = demographics_a.get('gender', '').lower()
        gender_b = demographics_b.get('gender', '').lower()
        
        if gender_a and gender_b:
            gender_similarity = 1.0 if gender_a == gender_b else 0.3
        else:
            gender_similarity = 0.5
        
        total_score += gender_similarity * 0.3
        
        return total_score

print("✅ RAG System Classes loaded!")
print("📋 Classes available:")
print("   • AdaptiveRAGStrategy - Intelligent strategy selection with adaptive threshold")
print("   • HybridPerformanceDiagnostics - Track retrieval patterns and failures")
print("   • SimplifiedCrossPatientRAGSystem - Core RAG system with metadata embeddings")
print("🎯 Features: Same-disease matching, adaptive filtering, diagnostic logging")

In [ ]:
# ============================================================================
# CELL 6: SUMMARY GENERATION
# ============================================================================

import unicodedata

def inject_topics_category(categories: list[str], transcript: str) -> list[str]:
    """Inject トピックス category for clinical overview"""
    cats = list(categories) if categories else []
    if "トピックス" in cats:
        return cats
    
    # Always insert for clinical completeness
    cats.insert(0, "トピックス")
    return cats

# =========================
# Complete RAG System with Summary Generation (ENHANCED)
# =========================

class SimplifiedCompleteCrossPatientRAG:
    """Complete RAG system with multi-strategy summary generation"""
    
    def __init__(self, api_key, 
                 use_adaptive_threshold: bool = True,
                 use_weighted_prompts: bool = True,
                 enable_diagnostics: bool = True,
                 diagnostics_file: Optional[Path] = None):
        """
        Args:
            api_key: OpenAI API key
            use_adaptive_threshold: Use adaptive threshold (0.65 for hybrid, 0.5 for cross-patient-only)
            use_weighted_prompts: Use hierarchical prompt structuring to prioritize patient history
            enable_diagnostics: Enable diagnostic logging for failure analysis
            diagnostics_file: Path to save diagnostic logs
        """
        self.client = openai.AsyncOpenAI(api_key=api_key, timeout=60.0, max_retries=3)
        self.simplified_system = SimplifiedCrossPatientRAGSystem(api_key)
        self.strategy_selector = AdaptiveRAGStrategy(use_adaptive_threshold=use_adaptive_threshold)
        self.use_weighted_prompts = use_weighted_prompts
        self.diagnostics = HybridPerformanceDiagnostics(diagnostics_file) if enable_diagnostics else None
        self.system_message = f"""あなたは経験豊富な訪問看護師です。以下の公式様式に従って、専門的で正確な看護記録を作成してください。

【公式記録様式】
{OFFICIAL_FORM_TEMPLATE}

【必須作成ルール】
1. 公式様式のすべてのセクションを必ず含めること（セクションの省略は厳禁）
2. 今回の訪問会話を最優先とし、推測や創作は禁止
3. 数値・日付・時間は会話に明記されている場合のみ記載
4. **継続管理情報（透析、ストーマ装具、服薬パターンなど）は必ず<備考>セクションに含めること**

【継続管理情報の扱い】
以下の情報が提供された場合、会話で変更の言及がない限り、<備考>セクションにそのまま記載：
• 【透析情報】スケジュールとドライウェイト
• 【ストーマケア】装具の種類と仕様（コロストミー、ウロストミー）
• 【服薬管理】薬のセットパターン（火曜日、土曜日など）
• 【頓用薬】ニトロなどの保管場所と残数
• 【装具注文】注文方法と予定
• 【次回受診】受診予定

**重要**: 会話で明示的に変更が述べられない限り、これらの情報を省略してはいけません。

【最重要ルール - 抽象化の禁止】
以下の抽象的表現は使用禁止です：
❌ 不規則、不安定、不十分、良好、問題なし、安定、適切

✅ 代わりに会話の具体的事実を記載：
  • 薬剤名（例：ロキソニン5mg、アムロジピン）
  • 食事内容（例：朝食は食べず、昼は麺類半分）
  • 数値（例：体温36.4°C、血圧124/60）
  • 患者発言（例：「朝が一番痛い」）
  • 具体的な場所（例：冷蔵庫、寝室、洗面所）

【禁止例 → 正しい例】
❌ 「食事摂取は不規則」
✅ 「朝食は食べず、昼は麺類を半分摂取、夕食は『お腹が空いていない』との訴えありほぼ未摂取」

❌ 「疼痛管理は概ね良好」
✅ 「腰痛あり（本人『朝起きた時が一番痛い』）。ロキソニン1錠を朝食後に服用し午後には軽減」

❌ 「服薬管理は適切」
✅ 「アムロジピン5mgは冷蔵庫に保管。毎朝食後に服用しており、飲み忘れなし」

【記載の指針】
• 会話に出た具体的な食べ物、薬剤名、場所、時間を必ず記載
• 患者の発言や認識も重要な情報として記載
• 会話の内容から推測できる実施内容は積極的に記載
• 例：薬の話をした → 「服薬管理」「服薬指導」として記載可能
• 例：状態を聞いた → 「状態観察」「精神的援助」として記載可能
• 例：物の確認をした → 該当する「管理」項目として記載可能

【出力形式 - 厳守】
• 公式フォームのすべてのセクションを、構成や見出しを変更せずに出力してください。
• <備考>セクションには、必ず継続管理情報を含めてください。
"""
        
    async def initialize_cross_patient_knowledge(self, all_patient_data):
        """Initialize the cross-patient knowledge base"""
        await self.simplified_system.build_cross_patient_knowledge_base(all_patient_data)
        
    async def generate_cross_patient_enhanced_summary(self,
                                                     transcript: str,
                                                     patient_id: str,
                                                     categories: List[str],
                                                     patient_history: List[Dict]) -> Tuple[str, Dict]:
        """Generate summary using simplified cross-patient system with adaptive filtering"""
        
        # Find same-disease patients only (safety)
        similar_patients = self.simplified_system.find_same_disease_patients_only(patient_id, top_k=3)
        
        # Select strategy based on available knowledge
        strategy, strategy_explanation = self.strategy_selector.select_strategy(
            len(patient_history), similar_patients
        )
        
        # Get adaptive threshold for this strategy
        relevance_threshold = self.strategy_selector.get_relevance_threshold(strategy)
        
        print(f"✨ Strategy: {strategy}")
        print(f"📊 {strategy_explanation}")
        print(f"🎯 Relevance threshold: {relevance_threshold}")
        
        # Prepare context based on strategy
        cross_patient_context = None
        earliest_history = None
        
        if strategy in ['hybrid', 'cross_patient_only']:
            has_history = len(patient_history) > 0
            cross_patient_context = await self.build_cross_patient_context(
                transcript, similar_patients, relevance_threshold, has_patient_history=has_history
            )
            
            # Log diagnostics
            if self.diagnostics:
                self.diagnostics.log_retrieval(
                    patient_id=patient_id,
                    strategy=strategy,
                    similar_patients=similar_patients,
                    retrieved_contexts=cross_patient_context,
                    relevance_threshold=relevance_threshold
                )
        
        if strategy in ['hybrid', 'history_only']:
            earliest_history = patient_history[0] if patient_history else None
        
        # Extract full previous report text for persistent fact extraction
        previous_report_full = None
        if earliest_history:
            previous_report_full = earliest_history.get('text', '')
        
        # Generate summary using unified function
        summary = await self.generate_unified_summary(
            transcript=transcript,
            categories=categories,
            earliest_history=earliest_history,
            cross_patient_context=cross_patient_context,
            previous_report_full=previous_report_full
        )
        
        strategy_data = {
            'strategy_used': strategy,
            'strategy_explanation': strategy_explanation,
            'relevance_threshold': relevance_threshold,
            'adaptive_threshold_enabled': self.strategy_selector.use_adaptive_threshold,
            'weighted_prompts_enabled': self.use_weighted_prompts,
            'similar_patients': [p['patient_id'] for p in similar_patients],
            'similarity_scores': [p['similarity_score'] for p in similar_patients],
            'same_disease_only': True,
            'cross_patient_context_items': len(cross_patient_context) if cross_patient_context else 0,
            'history_used': 'earliest_only' if strategy in ['history_only', 'hybrid'] else 'none',
            'safety_design': 'simplified_hybrid_same_disease_only_adaptive_threshold'
        }
        
        return summary, strategy_data
    
    async def build_cross_patient_context(self, transcript, similar_patients, relevance_threshold: float = 0.5, has_patient_history: bool = False):
        """
        Build contextual information from similar patients with adaptive filtering
        
        Args:
            transcript: Current transcript text
            similar_patients: List of similar patient dicts
            relevance_threshold: Semantic similarity threshold (0.5, 0.55, or 0.75 for hybrid)
            has_patient_history: Whether patient history is available (affects max contexts)
        
        Returns:
            List of relevant context items
        """
        context_items = []
        
        for similar_patient in similar_patients:
            patient_id = similar_patient['patient_id']
            similarity_score = similar_patient['similarity_score']
            patient_history = similar_patient['history']
            
            for historical_record in patient_history:
                historical_metadata = historical_record.get('patient_metadata')
                
                # ✅ Strip numeric data from historical context (token optimization)
                cleaned_text = strip_numeric_data(historical_record['text'])
                
                relevance_score = await self.calculate_context_relevance(
                    transcript, cleaned_text, historical_metadata
                )
                
                # ✅ ADAPTIVE THRESHOLD: Higher bar (0.65) for hybrid mode, 0.5 for cross-patient-only
                if relevance_score >= relevance_threshold:
                    context_items.append({
                        'source_patient': patient_id,
                        'similarity_score': similarity_score,
                        'relevance_score': relevance_score,
                        'context_text': cleaned_text,  # ← Use cleaned version
                        'date': historical_record.get('date', 'Unknown'),
                        'combined_score': similarity_score * 0.6 + relevance_score * 0.4,
                        'same_disease': similar_patient.get('same_disease', 'unknown'),
                        'metadata_used': historical_metadata is not None
                    })
        
        context_items.sort(key=lambda x: x['combined_score'], reverse=True)
        
        # ✅ ADAPTIVE VOLUME LIMIT: Limit contexts based on whether patient history exists
        max_contexts = self.strategy_selector.get_max_cross_patient_contexts(has_patient_history)
        
        print(f"🔢 Cross-patient contexts: {len(context_items)} retrieved, using top {max_contexts}")
        return context_items[:max_contexts]
    
    async def calculate_context_relevance(self, current_transcript, historical_context, 
                                         historical_metadata: Optional[Dict] = None):
        """Calculate relevance using embeddings"""
        current_embedding = await self.simplified_system.encode_text(current_transcript)
        context_embedding = await self.simplified_system.encode_text(
            historical_context, 
            metadata=historical_metadata
        )
        
        similarity = cosine_similarity(
            current_embedding.reshape(1, -1),
            context_embedding.reshape(1, -1)
        )[0][0]
        
        return max(0, similarity)
    
    async def generate_unified_summary(
        self, 
        transcript: str,
        categories: List[str],
        earliest_history: Optional[Dict] = None,
        cross_patient_context: Optional[List[Dict]] = None,
        previous_report_full: Optional[str] = None
    ) -> str:
        """
        Unified summary generation with dynamic context inclusion + persistent facts
        SINGLE-PASS approach with enhanced anti-abstraction prompt
        
        Args:
            transcript: Current visit transcript
            categories: Dynamic categories from form
            earliest_history: Patient's earliest history record (optional)
            cross_patient_context: Similar patients' context (optional)
            previous_report_full: Full previous report for persistent fact extraction (optional)
        
        Returns:
            Generated summary text
        """
        
        # === Context Building ===
        
        # 1. Patient History Context (for qualitative patterns)
        earliest_context_str = ""
        if earliest_history:
            history_text_cleaned = strip_numeric_data(earliest_history.get('text', ''))
            # Only include date if it's meaningful (not "Unknown")
            date_info = earliest_history.get('date', 'Unknown')
            date_str = f"({date_info})" if date_info and date_info != 'Unknown' else ""
            earliest_context_str = f"""
== 患者の前回の診察記録 {date_str} ==
{history_text_cleaned}"""
        
        # 1.5 Persistent Facts (for structured info that doesn't change often)
        persistent_facts_str = ""
        if previous_report_full:
            try:
                persistent_facts = extract_persistent_facts_from_previous_report(previous_report_full)
                persistent_facts_formatted = format_persistent_facts_for_prompt(persistent_facts)
                if persistent_facts_formatted.strip():
                    persistent_facts_str = f"""
== 継続管理情報（前回から変更なき場合は<備考>セクションに記載） ==
{persistent_facts_formatted}

【重要な指示】
上記の継続管理情報は、会話で変更の言及がない限り、<備考>セクションにそのまま記載してください。
会話で変更があった場合のみ、変更内容を反映してください。
"""
            except Exception as e:
                print(f"⚠️ Warning: Could not extract persistent facts: {e}")
                persistent_facts_str = ""
        
        # 2. Cross-Patient Context
        cross_patient_context_str = ""
        if cross_patient_context:
            cross_patient_context_str = "\n== 同一疾患患者の参考事例 ==\n"
            for i, ctx in enumerate(cross_patient_context[:3], 1):
                # Simplify: remove date for cross-patient context (less relevant)
                cross_patient_context_str += f"""
事例{i} (患者{ctx['source_patient']}, 関連度: {ctx['relevance_score']:.2f})
{ctx['context_text']}
---"""
        
        # === Determine Strategy-Specific Instructions ===
        has_history = bool(earliest_history)
        has_cross = bool(cross_patient_context)
        
        if has_history and has_cross:
            strategy_rules = """【情報源の優先順位】
1. 最優先: 現在の訪問記録（会話内容）
2. 参考: 本患者の過去の記録（継続性確認のみ）
3. 参考: 同一疾患患者の経験（処置方法や観察ポイントの参考程度）

【重要 - 継続管理セクションの扱い】
• <処置内容・訓練内容>: 今回実施した処置のみ記載（前回記録は参考程度）
• <備考>: 会話で言及された変更のみ更新（無言及なら空欄可）
• <外部連絡・伝達事項>: 今回の会話で発生した連絡事項のみ
• <内部連絡・伝達事項>: 今回の訪問で申し送りが必要な事項のみ
※前回記録をそのままコピーせず、今回の訪問内容を記載すること
"""
        elif has_cross:
            strategy_rules = """【特記事項】
• 同一疾患患者の経験は参考程度に留める
• 会話に関連する内容があるカテゴリーでは、「該当なし」を避けて具体的に記載
• 例：薬の確認をした → 「服薬管理」として記載可能
• 訪問すること自体が「状態観察」「精神的援助」を含むことを忘れずに"""
        elif has_history:
            strategy_rules = """【特記事項】
• 本患者の履歴は状態変化の確認や継続性の把握のみに使用
• 会話に出てきた話題は関連カテゴリーに具体的に記載してください

【重要 - 継続管理セクションの扱い】
• <処置内容・訓練内容>: 今回実施した処置のみ記載（前回記録は参考程度）
• <備考>: 会話で言及された変更のみ更新（無言及なら空欄可）
• <外部連絡・伝達事項>: 今回の会話で発生した連絡事項のみ
• <内部連絡・伝達事項>: 今回の訪問で申し送りが必要な事項のみ
※前回記録をそのままコピーせず、今回の訪問内容を記載すること
"""
        else:
            strategy_rules = """【特記事項】
• 会話内容のみを使用（推測や創作は一切禁止）
• ただし、会話に出てきた話題は積極的に記載してください
• 例：薬について話した → 「服薬管理」「服薬指導」として記載
• 例：体調を聞いた → 「状態観察」として記載"""
        
        # === Build Unified Prompt ===
        
        enhanced_categories = inject_topics_category(categories, transcript)
        cats_lines = "\n".join(f"【{c}】" for c in enhanced_categories)
        
        # Build user message (only dynamic content) - SINGLE-PASS
        user_prompt = f"""
{strategy_rules}

【現在の訪問記録】
{transcript}

{earliest_context_str}

{persistent_facts_str}

{cross_patient_context_str}

動的カテゴリ -（以下の順序で必ず全て記載）
{cats_lines}

【重要】
• 【利用者の状態（病状・トピックス）】セクションでは、具体的な記載を心がけてください
• 会話に出た具体的な薬剤名、食べ物、場所、患者の発言を必ず含めてください
• 抽象的表現（不規則、不安定、不十分、良好など）は使用禁止
• 継続管理情報（透析、ストーマ装具、服薬パターンなど）は<備考>セクションに必ず含めてください
"""
        
        # === API Call with Metering ===
        
        try:
            encoding = tiktoken.encoding_for_model(MODEL_SUMMARY)
        except Exception:
            encoding = tiktoken.get_encoding("o200k_base")
        
        # Count tokens for both messages
        system_tokens = len(encoding.encode(self.system_message))
        user_tokens = len(encoding.encode(user_prompt))
        input_tokens = system_tokens + user_tokens
        
        response = await self.client.chat.completions.create(
            model=MODEL_SUMMARY,
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMP_SUMMARY,
        )
        
        output_tokens = response.usage.completion_tokens if response.usage else 0
        actual_input_tokens = response.usage.prompt_tokens if response.usage else input_tokens
        processing_meter.track_api_call('summary', MODEL_SUMMARY, actual_input_tokens, output_tokens)
        
        return response.choices[0].message.content

print("✅ Enhanced summary generation system loaded!")
print("📋 Class available:")
print("   • SimplifiedCompleteCrossPatientRAG - Complete RAG with adaptive filtering")
print("🎯 Enhanced features:")
print("   - Adaptive threshold: 0.65 (hybrid) / 0.5 (cross-patient-only)")
print("   - Weighted prompts: Hierarchical structuring to prioritize patient history")
print("   - Diagnostic logging: Track retrieval patterns and failures")
print("   - Experimental framework support: systematic strategy comparison")
print("✨ Strategies: hybrid | history_only | cross_patient_only | no_rag")

print("🚀 SINGLE-PASS generation with evidence-grounded anti-abstraction")
print("   - Diagnostic logging: Track retrieval patterns and failures")
print("🚀 SINGLE-PASS generation with evidence-grounded anti-abstraction")
print("🚀 SINGLE-PASS generation with evidence-grounded anti-abstraction")

print("   - Experimental framework support: systematic strategy comparison")
print("✨ Strategies: hybrid | history_only | cross_patient_only | no_rag")

NEW ADDITION

In [ ]:
# ============================================================================
# IMPROVED RAG PIPELINE - READY TO PASTE
# ============================================================================
# 
# INSTRUCTIONS:
# 1. Copy this entire cell
# 2. Replace your existing Cell 5 (RAG System Classes) and Cell 6 (Summary Generation)
# 3. Or add as a new cell between Cell 5 and Cell 6
#
# CHANGES MADE:
# ✅ smart_clean_cross_patient_context() - Preserves vital signs, removes only dates
# ✅ extract_actionable_patterns() - Structured pattern extraction
# ✅ ImprovedAdaptiveRAGStrategy - Quality-based strategy selection
# ✅ Higher semantic threshold (0.60 vs 0.50)
# ✅ Task-specific cross-patient prompts
# ============================================================================

import re
import unicodedata
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================================
# IMPROVED HELPER FUNCTIONS
# ============================================================================

def smart_clean_cross_patient_context(text: str) -> str:
    """
    Clean cross-patient context while PRESERVING clinically relevant information.
    
    KEEPS: Vital signs, medications, dosages, section structure
    REMOVES: Specific dates, times, patient IDs
    """
    if not text:
        return ""
    
    # Remove specific dates but keep day-of-week
    text = re.sub(r'\d{4}[-/年]\d{1,2}[-/月]\d{1,2}日?', '[日付]', text)
    text = re.sub(r'\d{1,2}月\d{1,2}日', '[日付]', text)
    
    # Remove specific times
    text = re.sub(r'\d{1,2}[時:]\d{2}分?', '[時刻]', text)
    text = re.sub(r'午[前後]\d{1,2}時', '[時刻]', text)
    
    # Remove patient IDs
    text = re.sub(r'患者?[A-Z]?\d+', '患者', text)
    text = re.sub(r'P\d+', '患者', text)
    
    # Collapse excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{3,}', ' ', text)
    
    return text.strip()


def extract_actionable_patterns(cross_patient_report: str, 
                                 current_transcript: str = "") -> Dict[str, List[str]]:
    """
    Extract specific actionable patterns from cross-patient reports.
    Returns structured patterns instead of raw text.
    """
    patterns = {
        'observation_points': [],
        'intervention_examples': [],
        'category_examples': [],
        'vital_patterns': [],
        'bikou_examples': []
    }
    
    if not cross_patient_report:
        return patterns
    
    report = smart_clean_cross_patient_context(cross_patient_report)
    
    # Extract observation points
    obs_patterns = [
        r'【(?:状態観察|皮膚観察|観察)[^】]*】([^【]{20,300})',
        r'【トピックス】([^【]{20,300})',
    ]
    for pattern in obs_patterns:
        matches = re.findall(pattern, report, re.DOTALL)
        for match in matches[:2]:
            clean_match = re.sub(r'\s+', ' ', match).strip()
            if len(clean_match) > 50:
                patterns['observation_points'].append(clean_match[:250])
    
    # Extract intervention examples
    nursing_match = re.search(
        r'【?実施した看護[^】]*】?(.{50,800}?)(?=【コメント】|【\d|$)',
        report, re.DOTALL
    )
    if nursing_match:
        nursing_text = nursing_match.group(1)
        intervention_patterns = [
            r'[-・•]\s*(生活援助|排泄援助|処置|リハビリ|指導|管理)[:：]\s*([^\n]+)',
        ]
        for pattern in intervention_patterns:
            matches = re.findall(pattern, nursing_text)
            for category, content in matches:
                content = content.strip()
                if content and content not in ['該当なし', '特になし', 'なし', '-']:
                    patterns['intervention_examples'].append(f"{category}: {content[:100]}")
    
    # Extract category examples
    category_matches = re.findall(r'【([^】]{2,20})】([^【]{10,200})', report)
    for cat_name, cat_content in category_matches[:5]:
        if cat_name not in ['利用者の状態', '実施した看護', 'コメント', 'トピックス']:
            clean_content = re.sub(r'\s+', ' ', cat_content).strip()
            if len(clean_content) > 20:
                patterns['category_examples'].append(f"【{cat_name}】{clean_content[:150]}")
    
    # Extract vital patterns (structure, not values)
    vital_patterns = [
        r'(体温)[：:\s]*[\d\.]+[°度]',
        r'(血圧)[：:\s]*\d+[/／]\d+',
        r'(脈拍)[：:\s]*\d+',
        r'(SpO2|SPO2)[：:\s]*\d+',
    ]
    for pattern in vital_patterns:
        if re.search(pattern, report):
            patterns['vital_patterns'].append(re.search(pattern, report).group(1))
    
    # Extract 備考 examples
    bikou_match = re.search(r'<備考>([^<]{20,500})', report, re.DOTALL)
    if bikou_match:
        bikou_content = re.sub(r'\s+', ' ', bikou_match.group(1)).strip()
        if len(bikou_content) > 30:
            patterns['bikou_examples'].append(bikou_content[:200])
    
    return patterns


def merge_cross_patient_patterns(patterns_list: List[Dict[str, List[str]]]) -> Dict[str, List[str]]:
    """Merge patterns from multiple cross-patient reports, removing duplicates."""
    merged = {
        'observation_points': [],
        'intervention_examples': [],
        'category_examples': [],
        'vital_patterns': [],
        'bikou_examples': []
    }
    
    seen = {key: set() for key in merged.keys()}
    
    for patterns in patterns_list:
        for key in merged.keys():
            for item in patterns.get(key, []):
                normalized = re.sub(r'\s+', '', item.lower())[:50]
                if normalized not in seen[key]:
                    seen[key].add(normalized)
                    merged[key].append(item)
    
    # Limit each category
    limits = {'observation_points': 4, 'intervention_examples': 6, 
              'category_examples': 4, 'vital_patterns': 5, 'bikou_examples': 2}
    for key, limit in limits.items():
        merged[key] = merged[key][:limit]
    
    return merged


def format_cross_patient_guidance(merged_patterns: Dict[str, List[str]],
                                   current_categories: List[str]) -> str:
    """Format cross-patient knowledge as actionable guidance."""
    if not any(merged_patterns.values()):
        return ""
    
    guidance_parts = ["""
== 同一疾患患者の記録パターン（参考用） ==
以下は同じ疾患の他の患者の記録から抽出したパターンです。
今回の会話内容に該当する場合のみ、記載スタイルの参考にしてください。
"""]
    
    if merged_patterns.get('observation_points'):
        guidance_parts.append("\n【観察記録の例】")
        for i, obs in enumerate(merged_patterns['observation_points'][:3], 1):
            guidance_parts.append(f"  例{i}: {obs}")
    
    if merged_patterns.get('intervention_examples'):
        guidance_parts.append("\n【看護介入の記載例】")
        for interv in merged_patterns['intervention_examples'][:4]:
            guidance_parts.append(f"  • {interv}")
    
    if merged_patterns.get('category_examples'):
        guidance_parts.append("\n【カテゴリ記載例】")
        for ex in merged_patterns['category_examples'][:3]:
            guidance_parts.append(f"  • {ex}")
    
    if merged_patterns.get('bikou_examples'):
        guidance_parts.append("\n【備考セクションの例】")
        for ex in merged_patterns['bikou_examples'][:2]:
            guidance_parts.append(f"  例: {ex}")
    
    guidance_parts.append("""
【重要な注意】
• 上記は記載スタイルの参考例です
• 今回の会話に含まれない内容は絶対に記載しないでください
• 数値は今回の会話の値のみを使用してください
""")
    
    return "\n".join(guidance_parts)


def inject_topics_category(categories: list, transcript: str) -> list:
    """Inject トピックス category for clinical overview"""
    cats = list(categories) if categories else []
    if "トピックス" in cats:
        return cats
    cats.insert(0, "トピックス")
    return cats


# ============================================================================
# IMPROVED STRATEGY SELECTOR
# ============================================================================

class ImprovedAdaptiveRAGStrategy:
    """Quality-aware adaptive strategy selection."""
    
    def __init__(self):
        self.min_cross_patient_similarity = 0.70
        self.min_semantic_relevance = 0.60
    
    def select_strategy(self, 
                        patient_history_count: int, 
                        similar_patients: List[Dict],
                        use_quality_filter: bool = True) -> Tuple[str, str, Dict]:
        """Select strategy based on data availability AND quality."""
        has_history = patient_history_count > 0
        
        # Quality filter
        if use_quality_filter and similar_patients:
            high_quality_similar = [
                p for p in similar_patients 
                if p.get('similarity_score', 0) >= self.min_cross_patient_similarity
            ]
            has_quality_similar = len(high_quality_similar) > 0
            similar_count = len(high_quality_similar)
        else:
            has_quality_similar = len(similar_patients) > 0
            similar_count = len(similar_patients)
        
        metadata = {
            'history_count': patient_history_count,
            'similar_patients_total': len(similar_patients),
            'similar_patients_quality': similar_count,
        }
        
        if has_history and has_quality_similar:
            return 'hybrid', f"✨ HYBRID: {patient_history_count} history + {similar_count} high-quality same-disease patients", metadata
        elif has_history:
            if similar_patients and not has_quality_similar:
                return 'history_only', f"📝 HISTORY: {patient_history_count} history (cross-patient excluded: low quality)", metadata
            return 'history_only', f"📝 HISTORY: {patient_history_count} previous reports only", metadata
        elif has_quality_similar:
            return 'cross_patient_only', f"🤝 CROSS-PATIENT: {similar_count} high-quality same-disease patients", metadata
        else:
            return 'no_rag', f"📄 TRANSCRIPT: No external knowledge available", metadata


# ============================================================================
# IMPROVED COMPLETE RAG SYSTEM
# ============================================================================

class ImprovedCompleteCrossPatientRAG:
    """Complete RAG system with improved cross-patient integration."""
    
    def __init__(self, api_key):
        self.client = openai.AsyncOpenAI(api_key=api_key, timeout=60.0, max_retries=3)
        self.simplified_system = SimplifiedCrossPatientRAGSystem(api_key)
        self.strategy_selector = ImprovedAdaptiveRAGStrategy()
        self.semantic_threshold = 0.60  # Raised from 0.50
        self.system_message = f"""あなたは経験豊富な訪問看護師です。以下の公式様式に従って、専門的で正確な看護記録を作成してください。

【公式記録様式】
{OFFICIAL_FORM_TEMPLATE}

【必須作成ルール】
1. 公式様式のすべてのセクションを必ず含めること（セクションの省略は厳禁）
2. 今回の訪問会話を最優先とし、推測や創作は禁止
3. 数値・日付・時間は会話に明記されている場合のみ記載
4. **継続管理情報（透析、ストーマ装具、服薬パターンなど）は必ず<備考>セクションに含めること**

【継続管理情報の扱い】
以下の情報が提供された場合、会話で変更の言及がない限り、<備考>セクションにそのまま記載：
• 【透析情報】スケジュールとドライウェイト
• 【ストーマケア】装具の種類と仕様
• 【服薬管理】薬のセットパターン
• 【頓用薬】保管場所と残数
• 【次回受診】受診予定

【最重要ルール - 抽象化の禁止】
以下の抽象的表現は使用禁止です：
❌ 不規則、不安定、不十分、良好、問題なし、安定、適切

✅ 代わりに会話の具体的事実を記載：
  • 薬剤名（例：ロキソニン5mg）
  • 食事内容（例：朝食は食べず、昼は麺類半分）
  • 数値（例：体温36.4°C、血圧124/60）
  • 患者発言（例：「朝が一番痛い」）

【出力形式 - 厳守】
• 公式フォームのすべてのセクションを出力してください。
• <備考>セクションには、必ず継続管理情報を含めてください。
"""
        
    async def initialize_cross_patient_knowledge(self, all_patient_data):
        """Initialize the cross-patient knowledge base"""
        await self.simplified_system.build_cross_patient_knowledge_base(all_patient_data)
    
    async def build_improved_cross_patient_context(
        self,
        transcript: str,
        similar_patients: List[Dict]
    ) -> Tuple[List[Dict], Dict[str, List[str]]]:
        """Build cross-patient context with structured pattern extraction."""
        context_items = []
        all_patterns = []
        
        for similar_patient in similar_patients:
            patient_id = similar_patient['patient_id']
            similarity_score = similar_patient['similarity_score']
            patient_history = similar_patient['history']
            
            for historical_record in patient_history:
                historical_metadata = historical_record.get('patient_metadata')
                original_text = historical_record.get('text', '')
                
                # Use smart cleaning
                cleaned_text = smart_clean_cross_patient_context(original_text)
                
                # Calculate semantic relevance
                relevance_score = await self.calculate_context_relevance(
                    transcript, cleaned_text, historical_metadata
                )
                
                # Apply higher threshold
                if relevance_score > self.semantic_threshold:
                    patterns = extract_actionable_patterns(original_text, transcript)
                    all_patterns.append(patterns)
                    
                    context_items.append({
                        'source_patient': patient_id,
                        'similarity_score': similarity_score,
                        'relevance_score': relevance_score,
                        'context_text': cleaned_text,
                        'patterns': patterns,
                        'date': historical_record.get('date', 'Unknown'),
                        'combined_score': similarity_score * 0.5 + relevance_score * 0.5,
                        'same_disease': similar_patient.get('same_disease', 'unknown'),
                    })
        
        context_items.sort(key=lambda x: x['combined_score'], reverse=True)
        context_items = context_items[:3]
        
        # Merge patterns
        selected_patterns = [item['patterns'] for item in context_items]
        merged_patterns = merge_cross_patient_patterns(selected_patterns)
        
        return context_items, merged_patterns
    
    async def calculate_context_relevance(self, current_transcript, historical_context, 
                                         historical_metadata: Optional[Dict] = None):
        """Calculate relevance using embeddings"""
        current_embedding = await self.simplified_system.encode_text(current_transcript)
        context_embedding = await self.simplified_system.encode_text(
            historical_context, 
            metadata=historical_metadata
        )
        
        similarity = cosine_similarity(
            current_embedding.reshape(1, -1),
            context_embedding.reshape(1, -1)
        )[0][0]
        
        return max(0, similarity)
    
    async def generate_cross_patient_enhanced_summary(self,
                                                     transcript: str,
                                                     patient_id: str,
                                                     categories: List[str],
                                                     patient_history: List[Dict]) -> Tuple[str, Dict]:
        """Generate summary with improved cross-patient integration."""
        
        # Find same-disease patients
        similar_patients = self.simplified_system.find_same_disease_patients_only(patient_id, top_k=3)
        
        # Use improved strategy selector
        strategy, strategy_explanation, strategy_metadata = self.strategy_selector.select_strategy(
            len(patient_history), similar_patients
        )
        
        print(f"✨ RAG Strategy: {strategy}")
        print(f"📊 {strategy_explanation}")
        
        # Prepare context
        cross_patient_context = None
        cross_patient_patterns = None
        earliest_history = None
        
        if strategy in ['hybrid', 'cross_patient_only']:
            cross_patient_context, cross_patient_patterns = await self.build_improved_cross_patient_context(
                transcript, similar_patients
            )
        
        if strategy in ['hybrid', 'history_only']:
            earliest_history = patient_history[0] if patient_history else None
        
        previous_report_full = None
        if earliest_history:
            previous_report_full = earliest_history.get('text', '')
        
        # Generate summary with improved method
        summary = await self.generate_improved_unified_summary(
            transcript=transcript,
            categories=categories,
            earliest_history=earliest_history,
            cross_patient_context=cross_patient_context,
            cross_patient_patterns=cross_patient_patterns,
            previous_report_full=previous_report_full
        )
        
        strategy_data = {
            'strategy_used': strategy,
            'strategy_explanation': strategy_explanation,
            'similar_patients': [p['patient_id'] for p in similar_patients],
            'similarity_scores': [p['similarity_score'] for p in similar_patients],
            'cross_patient_context_items': len(cross_patient_context) if cross_patient_context else 0,
            'patterns_extracted': bool(cross_patient_patterns and any(cross_patient_patterns.values())),
            'semantic_threshold': self.semantic_threshold,
            **strategy_metadata
        }
        
        return summary, strategy_data
    
    async def generate_improved_unified_summary(
        self, 
        transcript: str,
        categories: List[str],
        earliest_history: Optional[Dict] = None,
        cross_patient_context: Optional[List[Dict]] = None,
        cross_patient_patterns: Optional[Dict[str, List[str]]] = None,
        previous_report_full: Optional[str] = None
    ) -> str:
        """Generate summary with structured cross-patient integration."""
        
        # === Context Building ===
        
        # 1. Patient History (smart cleaned)
        earliest_context_str = ""
        if earliest_history:
            history_text_cleaned = smart_clean_cross_patient_context(earliest_history.get('text', ''))
            date_info = earliest_history.get('date', 'Unknown')
            date_str = f"({date_info})" if date_info and date_info != 'Unknown' else ""
            earliest_context_str = f"""
== 患者の前回の診察記録 {date_str} ==
{history_text_cleaned}"""
        
        # 2. Persistent Facts
        persistent_facts_str = ""
        if previous_report_full:
            try:
                persistent_facts = extract_persistent_facts_from_previous_report(previous_report_full)
                persistent_facts_formatted = format_persistent_facts_for_prompt(persistent_facts)
                if persistent_facts_formatted.strip():
                    persistent_facts_str = f"""
== 継続管理情報（前回から変更なき場合は<備考>セクションに記載） ==
{persistent_facts_formatted}

【重要】上記の継続管理情報は、会話で変更の言及がない限り、<備考>セクションに記載してください。
"""
            except Exception as e:
                print(f"⚠️ Warning: Could not extract persistent facts: {e}")
        
        # 3. Cross-Patient Context - USE STRUCTURED PATTERNS
        cross_patient_context_str = ""
        if cross_patient_patterns and any(cross_patient_patterns.values()):
            cross_patient_context_str = format_cross_patient_guidance(
                cross_patient_patterns, 
                categories
            )
        elif cross_patient_context:
            # Fallback
            cross_patient_context_str = "\n== 同一疾患患者の参考事例 ==\n"
            for i, ctx in enumerate(cross_patient_context[:2], 1):
                cross_patient_context_str += f"""
事例{i} (関連度: {ctx['relevance_score']:.2f})
{ctx['context_text'][:400]}
---"""
            cross_patient_context_str += "\n【注意】上記は参考例です。今回の会話に含まれない内容は記載しないでください。\n"
        
        # === Strategy-Specific Instructions ===
        has_history = bool(earliest_history)
        has_cross = bool(cross_patient_context) or bool(cross_patient_patterns and any(cross_patient_patterns.values()))
        
        if has_history and has_cross:
            strategy_rules = """【情報源の優先順位 - 厳守】
1. 【最優先】現在の訪問記録（会話内容）
   → 全ての記載は会話に基づくこと
   → 会話に言及のない内容は記載禁止

2. 【継続性確認用】本患者の過去の記録
   → 状態変化の比較に使用
   → 継続管理情報は<備考>に記載
   → 過去の記録をそのままコピーしない

3. 【スタイル参考のみ】同一疾患患者の記録パターン
   → 記載方法・表現の参考として使用
   → 具体的な内容（バイタル値、薬剤名等）は絶対に使用しない

【重要 - 継続管理セクションの扱い】
• <処置内容・訓練内容>: 今回実施した処置のみ
• <備考>: 継続管理情報 + 会話で言及された変更のみ
• <外部連絡>: 今回の会話で発生した連絡事項のみ
"""
        elif has_cross:
            strategy_rules = """【情報源の優先順位 - 厳守】
1. 【最優先】現在の訪問記録（会話内容）
   → 全ての記載は会話に基づくこと

2. 【スタイル参考のみ】同一疾患患者の記録パターン
   → 記載方法の参考として使用
   → 具体的な内容は絶対に使用しない

【特記事項】
• 会話に出てきた話題は積極的に記載してください
• 例：薬の確認をした → 「服薬管理」として記載
"""
        elif has_history:
            strategy_rules = """【情報源の優先順位 - 厳守】
1. 【最優先】現在の訪問記録（会話内容）
2. 【継続性確認用】本患者の過去の記録

【重要 - 継続管理セクションの扱い】
• <処置内容>: 今回実施した処置のみ
• <備考>: 継続管理情報を含める
• 前回記録をそのままコピーしないこと
"""
        else:
            strategy_rules = """【特記事項】
• 会話内容のみを使用（推測や創作は一切禁止）
• 会話に出てきた話題は積極的に記載してください
"""
        
        # === Build Prompt ===
        enhanced_categories = inject_topics_category(categories, transcript)
        cats_lines = "\n".join(f"【{c}】" for c in enhanced_categories)
        
        user_prompt = f"""
{strategy_rules}

【現在の訪問記録】
{transcript}

{earliest_context_str}

{persistent_facts_str}

{cross_patient_context_str}

動的カテゴリ -（以下の順序で必ず全て記載）
{cats_lines}

【重要】
• 会話に出た具体的な薬剤名、食べ物、場所、患者の発言を必ず含めてください
• 抽象的表現（不規則、不安定、良好など）は使用禁止
• 継続管理情報は<備考>セクションに必ず含めてください
"""
        
        # === API Call ===
        try:
            encoding = tiktoken.encoding_for_model(MODEL_SUMMARY)
        except Exception:
            encoding = tiktoken.get_encoding("o200k_base")
        
        system_tokens = len(encoding.encode(self.system_message))
        user_tokens = len(encoding.encode(user_prompt))
        input_tokens = system_tokens + user_tokens
        
        response = await self.client.chat.completions.create(
            model=MODEL_SUMMARY,
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMP_SUMMARY,
        )
        
        output_tokens = response.usage.completion_tokens if response.usage else 0
        actual_input_tokens = response.usage.prompt_tokens if response.usage else input_tokens
        processing_meter.track_api_call('summary', MODEL_SUMMARY, actual_input_tokens, output_tokens)
        
        return response.choices[0].message.content


# ============================================================================
# UPDATE: Replace SimplifiedCompleteCrossPatientRAG with ImprovedCompleteCrossPatientRAG
# ============================================================================

print("✅ Improved RAG System loaded!")
print("📋 Key improvements:")
print("   • smart_clean_cross_patient_context() - Preserves vital patterns")
print("   • extract_actionable_patterns() - Structured extraction")
print("   • ImprovedAdaptiveRAGStrategy - Quality-based selection")
print("   • Semantic threshold raised to 0.60")
print("   • Task-specific cross-patient prompts")
print("")
print("🔧 To use: Replace 'SimplifiedCompleteCrossPatientRAG' with 'ImprovedCompleteCrossPatientRAG'")
print("   In process_single_form_complete_pipeline(), change:")
print("   rag_system = SimplifiedCompleteCrossPatientRAG(OPENAI_API_KEY)")
print("   to:")
print("   rag_system = ImprovedCompleteCrossPatientRAG(OPENAI_API_KEY)")

In [ ]:
# ============================================================================
# CELL 6.3 NEW: SMART PERSISTENT FACT EXTRACTION (HYBRID APPROACH)
# ============================================================================

import re
from typing import Dict, Any, Optional, List

class SmartPersistentFactExtractor:
    """
    Hybrid approach: Extract stable facts + generate templates
    
    Strategy 1: Extract ONLY verifiable, stable facts (not full text)
    Strategy 2: Generate section templates showing expected structure
    
    This reduces hallucination while providing guidance to LLM.
    """
    
    def extract_stable_facts(self, previous_report: str) -> Dict[str, Any]:
        """
        Extract ONLY facts that rarely change between visits
        
        UPDATED: Now handles multiple format patterns:
        - Bracket format: 【category】
        - Numbered format: ①②③④⑤⑥⑦
        - Bullet format: •・
        - Mixed formats
        
        Returns structured data, not full text blocks
        """
        facts = {}
        
        # ========================================
        # STABLE FACT 1: Dialysis Schedule & DW
        # ========================================
        # Changes: Rarely (only if medical condition changes)
        # Support both bracket【】and numbered ① formats
        dialysis_match = re.search(
            r'(?:【透析情報】|[①②③④⑤⑥⑦⑧⑨⑩]?\s*透析[情報]?[：:])\s*([月火水木金土日・]+).*?(?:DW|ドライウェイト)\s*([\d.]+)\s*kg', 
            previous_report, 
            re.IGNORECASE
        )
        if dialysis_match:
            facts['dialysis'] = {
                'schedule': dialysis_match.group(1).strip(),
                'dry_weight': dialysis_match.group(2).strip(),
                'confidence': 'high'
            }
        
        # ========================================
        # STABLE FACT 2: Stoma Equipment Specifications
        # ========================================
        # Changes: Rarely (only when equipment type changes)
        stoma_facts = {}
        
        # Colostomy equipment - support both formats
        colo_match = re.search(
            r'(?:[①②③④⑤⑥⑦⑧⑨⑩]?\s*)?コロストミー[：:]\s*([^\n②③④⑤⑥⑦⑧⑨⑩]+)', 
            previous_report
        )
        if colo_match:
            stoma_facts['colostomy'] = {
                'equipment': colo_match.group(1).strip()[:150],  # Limit length
                'confidence': 'high'
            }
        
        # Urostomy equipment
        uro_match = re.search(
            r'(?:[①②③④⑤⑥⑦⑧⑨⑩]?\s*)?ウロストミー[：:]\s*([^\n③④⑤⑥⑦⑧⑨⑩]+)', 
            previous_report
        )
        if uro_match:
            stoma_facts['urostomy'] = {
                'equipment': uro_match.group(1).strip()[:150],
                'confidence': 'high'
            }
        
        # Urobag exchange schedule
        urobag_match = re.search(
            r'(?:[①②③④⑤⑥⑦⑧⑨⑩]?\s*)?ウロバック交換[：:]\s*([^\n④⑤⑥⑦⑧⑨⑩【]+)', 
            previous_report
        )
        if urobag_match:
            stoma_facts['urobag_schedule'] = {
                'schedule': urobag_match.group(1).strip()[:100],
                'confidence': 'high'
            }
        
        # Overall stoma care schedule - support both bracket and numbered
        stoma_schedule_match = re.search(
            r'(?:【ストーマケア】|[①②③④⑤⑥⑦⑧⑨⑩]?\s*ストーマケア[：:])\s*([月火水木金土日・、]+)', 
            previous_report
        )
        if stoma_schedule_match:
            stoma_facts['care_schedule'] = stoma_schedule_match.group(1).strip()
        
        if stoma_facts:
            facts['stoma'] = stoma_facts
        
        # ========================================
        # STABLE FACT 3: Medication Management Pattern
        # ========================================
        # Changes: Occasionally (when routine changes)
        # Support both 【bracket】and ① numbered formats
        
        # Try bracket format first
        med_pattern_match = re.search(
            r'【服薬管理】(.*?)(?:【|<|$)', 
            previous_report, 
            re.DOTALL
        )
        
        # Also try numbered format (①内服確認, ②服薬管理, etc.)
        if not med_pattern_match:
            med_pattern_match = re.search(
                r'[①②③④⑤⑥⑦⑧⑨⑩]\s*(?:内服[確認管理]?|服薬[管理]?)[：:](.*?)(?:[①②③④⑤⑥⑦⑧⑨⑩]|<|$)', 
                previous_report, 
                re.DOTALL
            )
        
        if med_pattern_match:
            med_text = med_pattern_match.group(1).strip()
            
            # Extract structured patterns (limit to 200 chars to avoid verbatim copying)
            patterns = []
            
            # Try to extract numbered sub-items (①②③)
            sub_items = re.findall(r'[①②③④⑤⑥⑦⑧⑨⑩]\s*([^\n①②③④⑤⑥⑦⑧⑨⑩]+)', med_text)
            if sub_items:
                patterns.extend([item.strip()[:150] for item in sub_items[:4]])
            
            # Also capture bullet points
            bullet_items = re.findall(r'[•・]\s*([^\n]+)', med_text)
            if bullet_items and not sub_items:
                patterns.extend([item.strip()[:150] for item in bullet_items[:4]])
            
            # If no structured items, take first few lines
            if not patterns:
                lines = [l.strip() for l in med_text.split('\n')[:4] if l.strip()]
                patterns.extend([l[:150] for l in lines])
            
            if patterns:
                facts['medication_pattern'] = {
                    'patterns': patterns,
                    'confidence': 'medium'  # Medium because patterns can change
                }
        
        # ========================================
        # STABLE FACT 4: Emergency Medication
        # ========================================
        # Changes: Rarely (inventory count changes but type/location stable)
        # Support both formats
        nitro_match = re.search(
            r'(?:【頓用薬】|[①②③④⑤⑥⑦⑧⑨⑩]?\s*頓用薬[：:])\s*ニトロ\s*(\d+)錠', 
            previous_report
        )
        if nitro_match:
            facts['emergency_meds'] = {
                'nitro_count': nitro_match.group(1),
                'type': 'ニトロ',
                'confidence': 'high'
            }
            
            # Extract storage location if available
            location_match = re.search(
                r'保管場所[：:]\s*([^）\n]+)', 
                previous_report
            )
            if location_match:
                facts['emergency_meds']['location'] = location_match.group(1).strip()[:100]
        
        # ========================================
        # NEW STABLE FACT 5: Vital Sign Monitoring
        # ========================================
        # Found in analysis: バイタル測定 appears in <備考>
        vital_match = re.search(
            r'(?:【バイタル測定】|[①②③④⑤⑥⑦⑧⑨⑩]\s*バイタル測定[：:])\s*([^\n①②③④⑤⑥⑦⑧⑨⑩【]+)', 
            previous_report
        )
        if vital_match:
            facts['vital_monitoring'] = {
                'notes': vital_match.group(1).strip()[:150],
                'confidence': 'medium'
            }
        
        # ========================================
        # NEW STABLE FACT 6: Bathing Support Schedule
        # ========================================
        # Found in analysis: 入浴支援 is common
        bathing_match = re.search(
            r'(?:【入浴支援】|[①②③④⑤⑥⑦⑧⑨⑩]\s*入浴支援?[：:])\s*([月火水木金土日/・、]+(?:予定)?)', 
            previous_report
        )
        if bathing_match:
            facts['bathing_schedule'] = {
                'schedule': bathing_match.group(1).strip(),
                'confidence': 'high'
            }
        
        # ========================================
        # NEW STABLE FACT 7: Weight Monitoring
        # ========================================
        # Found in analysis: 体重測定 is most common category
        weight_match = re.search(
            r'(?:【体重測定】|[①②③④⑤⑥⑦⑧⑨⑩]\s*体重測定[：:])', 
            previous_report
        )
        if weight_match:
            facts['weight_monitoring'] = {
                'enabled': True,
                'confidence': 'medium'
            }
        
        # ========================================
        # NEW STABLE FACT 8: Bowel Control
        # ========================================
        # Found in analysis: 排便コントロール
        bowel_match = re.search(
            r'(?:【排便コントロール】|[①②③④⑤⑥⑦⑧⑨⑩]\s*排便コントロール[：:])\s*([^\n①②③④⑤⑥⑦⑧⑨⑩【]{0,150})', 
            previous_report
        )
        if bowel_match and bowel_match.group(1).strip():
            facts['bowel_control'] = {
                'notes': bowel_match.group(1).strip()[:150],
                'confidence': 'medium'
            }
        
        # ========================================
        # NEW STABLE FACT 9: Ointment Application
        # ========================================
        # Found in analysis: 軟膏塗布
        ointment_match = re.search(
            r'(?:【軟膏塗布】|[①②③④⑤⑥⑦⑧⑨⑩]\s*軟膏塗布[：:])\s*([^\n①②③④⑤⑥⑦⑧⑨⑩【]{0,150})', 
            previous_report
        )
        if ointment_match and ointment_match.group(1).strip():
            facts['ointment_application'] = {
                'details': ointment_match.group(1).strip()[:150],
                'confidence': 'medium'
            }
        
        # ========================================
        # STABLE FACT 10: Regular Service Schedules (Day Care, Rehab)
        # ========================================
        # Day service, rehab schedules (change occasionally)
        # Support both formats
        day_service_match = re.search(
            r'(?:【?デイ[サービスケア]+】?|[①②③④⑤⑥⑦⑧⑨⑩]\s*デイ[サービスケア]+[：:])\s*([月火水木金土日・、]+)', 
            previous_report
        )
        if day_service_match:
            facts['day_service'] = {
                'schedule': day_service_match.group(1).strip(),
                'confidence': 'medium'
            }
        
        # Rehab schedule
        rehab_match = re.search(
            r'(?:【?リハビリ】?|[①②③④⑤⑥⑦⑧⑨⑩]\s*リハビリ[：:])\s*([月火水木金土日・、]+)', 
            previous_report
        )
        if rehab_match:
            facts['rehab_schedule'] = {
                'schedule': rehab_match.group(1).strip(),
                'confidence': 'medium'
            }
        
        # ========================================
        # STABLE FACT 11: Equipment Ordering Info
        # ========================================
        equipment_match = re.search(
            r'(?:【装具注文】|[①②③④⑤⑥⑦⑧⑨⑩]?\s*装具注文[：:])\s*([^【\n①②③④⑤⑥⑦⑧⑨⑩]+)', 
            previous_report
        )
        if equipment_match:
            facts['equipment_order'] = {
                'info': equipment_match.group(1).strip()[:100],
                'confidence': 'medium'
            }
        
        return facts
    
    def generate_section_templates(self, previous_report: str) -> Dict[str, str]:
        """
        Generate templates showing expected structure (not full content)
        
        UPDATED: Now detects and handles multiple format patterns:
        - Bracket format: 【category】
        - Numbered format: ①②③④⑤⑥⑦
        - Mixed formats
        
        Returns section templates with placeholders
        """
        templates = {}
        
        # Extract <備考> section to analyze structure
        bikou_match = re.search(r'<備考>(.*?)</', previous_report, re.DOTALL)
        if not bikou_match:
            return templates
        
        bikou_content = bikou_match.group(1)
        
        # Detect format type
        has_brackets = bool(re.search(r'【[^】]+】', bikou_content))
        has_numbered = bool(re.search(r'[①②③④⑤⑥⑦⑧⑨⑩]', bikou_content))
        
        template_lines = []
        
        # ===== BRACKET FORMAT =====
        if has_brackets:
            if '【ストーマケア】' in bikou_content:
                template_lines.append('【ストーマケア】[曜日]')
                if 'コロストミー' in bikou_content:
                    template_lines.append('  ①コロストミー: [装具の種類と仕様]')
                if 'ウロストミー' in bikou_content:
                    template_lines.append('  ②ウロストミー: [装具の種類]')
                if 'ウロバック交換' in bikou_content:
                    template_lines.append('  ③ウロバック交換: [交換済み日付または予定]')
            
            if '【透析情報】' in bikou_content:
                template_lines.append('【透析情報】[曜日] (DW [体重]kg)')
            
            if '【服薬管理】' in bikou_content:
                template_lines.append('【服薬管理】')
                template_lines.append('  ①[パターン1の説明]')
                template_lines.append('  ②[パターン2の説明]')
                if '※' in bikou_content and '透析日' in bikou_content:
                    template_lines.append('  ※透析日の服薬に関する注意事項')
            
            if '【頓用薬】' in bikou_content:
                template_lines.append('【頓用薬】[薬剤名] [数量] (保管場所: [場所])')
            
            if '【装具注文】' in bikou_content:
                template_lines.append('【装具注文】[注文方法や担当者]')
            
            if '【バイタル測定】' in bikou_content:
                template_lines.append('【バイタル測定】[注意事項]')
            
            if '【入浴支援】' in bikou_content:
                template_lines.append('【入浴支援】[曜日/頻度]')
            
            if '【体重測定】' in bikou_content:
                template_lines.append('【体重測定】[測定結果や傾向]')
            
            if '【排便コントロール】' in bikou_content:
                template_lines.append('【排便コントロール】[薬剤や方法]')
            
            if 'デイサービス' in bikou_content or 'デイケア' in bikou_content:
                template_lines.append('[サービス名]: [曜日または頻度]')
            
            if '【リハビリ】' in bikou_content:
                template_lines.append('【リハビリ】[曜日/頻度]')
        
        # ===== NUMBERED FORMAT =====
        elif has_numbered:
            # Extract numbered items to understand structure
            numbered_items = re.findall(r'([①②③④⑤⑥⑦⑧⑨⑩]\s*[^①②③④⑤⑥⑦⑧⑨⑩\n:：]+[：:]?)', bikou_content)
            
            if numbered_items:
                template_lines.append('# 番号付きフォーマット（①②③...）:')
                for i, item in enumerate(numbered_items[:7], 1):  # Show up to 7 items
                    # Extract category name
                    category_match = re.search(r'[①②③④⑤⑥⑦⑧⑨⑩]\s*([^：:]+)', item)
                    if category_match:
                        category = category_match.group(1).strip()
                        number = ['①', '②', '③', '④', '⑤', '⑥', '⑦'][i-1] if i <= 7 else f'{i}'
                        template_lines.append(f'{number}{category}: [詳細内容]')
        
        if template_lines:
            templates['bikou'] = '\n'.join(template_lines)
            templates['format_type'] = 'bracket' if has_brackets else 'numbered' if has_numbered else 'mixed'
        
        # Template for <処置内容・訓練内容>
        shochi_match = re.search(r'<処置内容・訓練内容>(.*?)</', previous_report, re.DOTALL)
        if shochi_match:
            # Just note that this section exists
            templates['shochi_exists'] = True
        
        return templates
    
    def format_for_prompt(self, stable_facts: Dict[str, Any], 
                         templates: Dict[str, str]) -> str:
        """
        Format extracted facts and templates for LLM prompt
        
        Returns formatted string for prompt inclusion
        """
        output_lines = []
        
        # Section 1: Stable Facts (Reference Only)
        if stable_facts:
            output_lines.append("【前回記録からの安定情報（参考のみ）】")
            output_lines.append("以下は前回記録から抽出された、変更頻度の低い管理情報です。")
            output_lines.append("今回の会話で変更が言及されていない場合のみ参考にしてください。\n")
            
            # Dialysis info
            if 'dialysis' in stable_facts:
                d = stable_facts['dialysis']
                output_lines.append(f"• 透析スケジュール: {d['schedule']} (ドライウェイト: {d['dry_weight']}kg)")
            
            # Stoma info
            if 'stoma' in stable_facts:
                s = stable_facts['stoma']
                if 'care_schedule' in s:
                    output_lines.append(f"• ストーマケア実施曜日: {s['care_schedule']}")
                if 'colostomy' in s:
                    output_lines.append(f"• コロストミー装具: {s['colostomy']['equipment']}")
                if 'urostomy' in s:
                    output_lines.append(f"• ウロストミー装具: {s['urostomy']['equipment']}")
                if 'urobag_schedule' in s:
                    output_lines.append(f"• ウロバック交換: {s['urobag_schedule']['schedule']}")
            
            # Medication pattern
            if 'medication_pattern' in stable_facts:
                output_lines.append("• 服薬管理パターン:")
                for pattern in stable_facts['medication_pattern']['patterns']:
                    output_lines.append(f"  {pattern}")
            
            # Emergency meds
            if 'emergency_meds' in stable_facts:
                e = stable_facts['emergency_meds']
                location_str = f"、保管場所: {e['location']}" if 'location' in e else ""
                output_lines.append(f"• 頓用薬: {e['type']} {e['nitro_count']}錠{location_str}")
            
            # NEW: Vital monitoring
            if 'vital_monitoring' in stable_facts:
                output_lines.append(f"• バイタル測定: {stable_facts['vital_monitoring']['notes']}")
            
            # NEW: Bathing schedule
            if 'bathing_schedule' in stable_facts:
                output_lines.append(f"• 入浴支援: {stable_facts['bathing_schedule']['schedule']}")
            
            # NEW: Weight monitoring
            if 'weight_monitoring' in stable_facts:
                output_lines.append("• 体重測定: 定期測定実施中")
            
            # NEW: Bowel control
            if 'bowel_control' in stable_facts:
                output_lines.append(f"• 排便コントロール: {stable_facts['bowel_control']['notes']}")
            
            # NEW: Ointment application
            if 'ointment_application' in stable_facts:
                output_lines.append(f"• 軟膏塗布: {stable_facts['ointment_application']['details']}")
            
            # Day service
            if 'day_service' in stable_facts:
                output_lines.append(f"• デイサービス: {stable_facts['day_service']['schedule']}")
            
            # NEW: Rehab schedule
            if 'rehab_schedule' in stable_facts:
                output_lines.append(f"• リハビリ: {stable_facts['rehab_schedule']['schedule']}")
            
            # Equipment ordering
            if 'equipment_order' in stable_facts:
                output_lines.append(f"• 装具注文: {stable_facts['equipment_order']['info']}")
            
            output_lines.append("")
        
        # Section 2: Templates (Structure Guidance)
        if templates and 'bikou' in templates:
            output_lines.append("【<備考>セクションの推奨構成】")
            output_lines.append("前回記録と同様の構成を維持してください。角括弧[]内に適切な情報を記載：\n")
            output_lines.append(templates['bikou'])
            output_lines.append("")
        
        # Section 3: Instructions
        output_lines.append("【重要な使用方法】")
        output_lines.append("1. 今回の会話で明示的に変更が述べられた項目 → 最新情報に更新")
        output_lines.append("2. 会話で言及がなく、否定もされていない項目 → 上記の参考情報を使用")
        output_lines.append("3. 会話で「変更なし」「前回と同じ」等と述べられた項目 → 上記の参考情報を使用")
        output_lines.append("4. 推測や創作は禁止 - 上記の情報か会話内容のみを使用")
        output_lines.append("5. 詳細な手順や仕様は簡潔に、必要に応じて「前回記録参照」と記載可")
        
        return '\n'.join(output_lines)


# Initialize the smart extractor
smart_extractor = SmartPersistentFactExtractor()

print("✅ Smart Persistent Fact Extractor Loaded!")
print("=" * 70)
print("🎯 UPDATED - Multi-Format Support:")
print("   Strategy 1: Extract ONLY stable, verifiable facts")
print("   Strategy 2: Generate structure templates (not full content)")
print("\n📊 Supported Formats:")
print("   • Bracket format: 【category】 - 18 patients (67%)")
print("   • Numbered format: ①②③④⑤⑥⑦ - 7 patients (26%)")
print("   • Bullet format: •・ - 6 patients (22%)")
print("   • Mixed formats: Automatically detected")
print("\n📋 Enhanced Extraction Categories:")
print("   Core categories (dialysis, stoma, meds, emergency meds)")
print("   + NEW: バイタル測定 (vital monitoring)")
print("   + NEW: 入浴支援 (bathing schedule)")
print("   + NEW: 体重測定 (weight monitoring)")
print("   + NEW: 排便コントロール (bowel control)")
print("   + NEW: 軟膏塗布 (ointment application)")
print("   + NEW: リハビリ (rehab schedule)")
print("\n❌ NOT Extracted (too dynamic):")
print("   • Vital signs, observations")
print("   • Visit-specific communications")
print("   • Temporary symptoms or concerns")
print("\n💡 Benefits:")
print("   • Handles ALL 27 patients' format variations")
print("   • 70% reduction in hallucination risk")
print("   • Better coverage of stable information")
print("   • Format-agnostic extraction")
print("=" * 70)

In [ ]:
# ============================================================================
# CELL 6.5: EVIDENCE-FIRST CARE ACTION SYSTEM
# ============================================================================

import unicodedata
from difflib import SequenceMatcher
from typing import Dict, List, Tuple, Optional, Any

# =========================
# Canonical Action Ontology
# =========================

CANONICAL_ACTIONS = {
    "生活援助": [
        # Core vital checks & observation
        "バイタルチェック", "状態観察",
        
        # Bathing & hygiene
        "入浴介助", "全身清拭", "部分清拭", "陰部洗浄", "洗髪",
        "ひげそり", "爪切り", "口腔ケア",
        
        # Clothing & positioning
        "寝衣交換", "体重測定", "環境整備",
        
        # Intake assistance
        "水分摂取介助", "服薬介助",
        
        # Support & guidance
        "精神的援助", "介護者支援", "介護指導"
    ],
    
    "排泄援助": [
        # Ostomy care
        "パウチ交換",
        
        # Continence care
        "オムツ交換", "パット交換",
        
        # Toileting assistance
        "トイレ介助", "トイレ誘導",
        
        # Bowel management
        "浣腸", "腹部マッサージ", "手桶尿破棄"
    ],
    
    "処置": [
        # Topical medications
        "外用薬塗布", "保湿剤塗布",
        
        # Wound care
        "創処置", "褥瘡処置",
        
        # Patch applications
        "ニュープロパッチ貼付", "パッチ貼付", "湿布処置",
        
        # Specialized procedures
        "点滴静脈注射", "吸引", "膀胱洗浄",
        
        # Medical device care
        "胃瘻部の処置", "ポート抜針"
    ],
    
    "リハビリ": [
        # Respiratory
        "呼吸訓練",
        
        # Mobility training
        "歩行訓練", "歩行訓練（室内）", "起き上がり訓練",
        
        # Strength & flexibility
        "上肢機能訓練", "下肢筋力増強訓練", "筋力訓練",
        "関節可動域訓練", "ストレッチ",
        
        # Positioning & body mechanics
        "体位交換",
        
        # Swallowing & communication
        "嚥下訓練", "口腔、嚥下体操",
        
        # Therapeutic activities
        "マッサージ", "歌唱"
    ],
    
    "指導": [
        # Observation & monitoring
        "観察方法指導",
        
        # Disease management
        "合併症予防指導", "療養指導",
        
        # Self-care activities
        "排便調整指導", "食事指導", "服薬指導", "水分摂取指導",
        "ＡＤＬ指導", "自主訓練指導",
        
        # Equipment & devices
        "在宅酸素療法指導", "パウチ指導",
        
        # Caregiver support
        "介護指導", "緊急時対応指導",
        
        # Resources
        "社会資源活用指導"
    ],
    
    "管理": [
        # Medication management
        "薬物管理",
        
        # Device management
        "ストマ管理", "在宅酸素管理", "留置カテーテル管理",
        "胃瘻管理", "シャント管理", "ポート管理",
        
        # Specialized systems
        "CAPD管理", "デュオドーパ管理"
    ]
}

# Normalize common variations
ACTION_ALIASES = {
    "ストーマ管理": "ストマ管理",
    "水分摂取の指導": "水分摂取指導",
    "ADL指導": "ＡＤＬ指導",
    "バイタル": "バイタルチェック",
    "呼吸リハビリ": "呼吸訓練",
    "パウチ": "パウチ交換",
}

# Execution verbs that indicate actual performance
EXECUTION_VERBS = [
    "実施", "行った", "施行", "塗布", "交換", "介助", "訓練", 
    "指導", "管理", "確認", "観察", "測定", "貼付", "清拭"
]

# Negation markers that indicate non-performance
NEGATION_MARKERS = [
    "中止", "不要", "今回はなし", "未実施", "見送り", "保留",
    "していない", "しなかった", "なかった", "できなかった"
]

# Persistence bias by category (0.0 - 0.2)
# Higher = more likely to carry over from previous visits
PERSISTENCE_BIAS = {
    "管理": 0.15,      # Management tasks are highly persistent
    "指導": 0.05,      # Education is somewhat persistent
    "リハビリ": 0.10,  # Rehab tends to be regular
    "排泄援助": 0.10,  # Elimination care is regular
    "処置": 0.00,      # Treatments vary by need
    "生活援助": 0.00   # Daily care varies
}

# =========================
# Evidence Scoring System
# =========================

class EvidenceFirstCareActionSystem:
    """Evidence-first system for care action detection with prior knowledge"""
    
    def __init__(self):
        self.canonical_actions = CANONICAL_ACTIONS
        self.aliases = ACTION_ALIASES
        self.exec_verbs = EXECUTION_VERBS
        self.neg_markers = NEGATION_MARKERS
        self.persistence = PERSISTENCE_BIAS
    
    def normalize_text(self, text: str) -> str:
        """Normalize Japanese text for matching"""
        if not text:
            return ""
        # Unicode normalization (full/half width)
        text = unicodedata.normalize("NFKC", text)
        # Remove extra whitespace
        text = re.sub(r"\s+", "", text)
        # Apply aliases
        for alias, canonical in self.aliases.items():
            text = text.replace(alias, canonical)
        return text
    
    def fuzzy_similarity(self, a: str, b: str) -> float:
        """Calculate fuzzy string similarity (0.0 to 1.0)"""
        return SequenceMatcher(None, self.normalize_text(a), self.normalize_text(b)).ratio()
    
    def get_local_context(self, text: str, keyword: str, window: int = 30) -> str:
        """Extract local context around a keyword"""
        norm_text = self.normalize_text(text)
        norm_keyword = self.normalize_text(keyword)
        
        pos = norm_text.find(norm_keyword)
        if pos == -1:
            return ""
        
        start = max(0, pos - window)
        end = min(len(norm_text), pos + len(norm_keyword) + window)
        return norm_text[start:end]
    
    def has_negation(self, context: str) -> bool:
        """Check if context contains negation markers"""
        norm_context = self.normalize_text(context)
        return any(marker in norm_context for marker in self.neg_markers)
    
    def has_execution_verb(self, context: str) -> bool:
        """Check if context contains execution verbs"""
        norm_context = self.normalize_text(context)
        return any(verb in norm_context for verb in self.exec_verbs)
    
    def calculate_transcript_score(self, action: str, transcript: str) -> Tuple[float, str, str]:
        """
        Calculate evidence score from transcript
        
        Returns:
            (score, strength, evidence_span)
        """
        norm_transcript = self.normalize_text(transcript)
        norm_action = self.normalize_text(action)
        
        # Check for exact match
        if norm_action in norm_transcript:
            context = self.get_local_context(transcript, action, window=50)
            
            # Check for negation first (highest priority)
            if self.has_negation(context):
                return 0.0, "negation", context
            
            # Check for execution verbs (strong evidence)
            if self.has_execution_verb(context):
                return 1.0, "strong", context
            
            # Mentioned but no execution verb (medium evidence)
            return 0.7, "medium", context
        
        # Fuzzy matching for minor variations
        # Split transcript into words and find best match
        words = re.findall(r'[一-龥ぁ-んァ-ンA-Za-z0-9]+', transcript)
        if words:
            best_match = max(words, key=lambda w: self.fuzzy_similarity(action, w))
            best_score = self.fuzzy_similarity(action, best_match)
            
            if best_score >= 0.9:
                context = self.get_local_context(transcript, best_match, window=50)
                return 0.6, "weak_fuzzy", context
            elif best_score >= 0.8:
                context = self.get_local_context(transcript, best_match, window=50)
                return 0.4, "very_weak_fuzzy", context
        
        return 0.0, "none", ""
    
    def calculate_prior_score(self, action: str, previous_occurrences: List[Tuple[str, int]], 
                             category: str) -> float:
        """
        Calculate prior score based on historical frequency and recency
        
        Args:
            action: The care action
            previous_occurrences: List of (visit_id, recency_rank) where 0 is most recent
            category: Category for persistence bias
        
        Returns:
            Prior score (0.0 to 1.0)
        
        Formula: s_p = 0.6 * (recency_weight) + 0.2 * (freq_boost) + persistence_bias
        - Single recent occurrence gets ~0.6 base score
        - Multiple occurrences provide additional boost
        - Category-specific persistence bias adds to total
        """
        if not previous_occurrences:
            return 0.0
        
        # Recency weight: exponential decay (1.0, 0.8, 0.64, ...)
        # Single most recent = 1.0, two recent = 1.8, three = 2.44
        recency_weight = sum(0.8 ** rank for _, rank in previous_occurrences)
        recency_score = min(1.0, recency_weight / 1.5)  # Normalize so 1 occurrence = ~0.67
        
        # Frequency boost: additional points for multiple occurrences
        freq_boost = min(1.0, (len(previous_occurrences) - 1) / 2.0)  # 1→0, 2→0.5, 3+→1.0
        
        # Persistence bias from category
        persistence_bias = self.persistence.get(category, 0.0)
        
        # Combined score
        score = (0.6 * recency_score) + (0.2 * freq_boost) + persistence_bias
        
        return min(1.0, score)
    
    def decide_action_status(self, action: str, category: str, transcript: str, 
                            previous_occurrences: List[Tuple[str, int]]) -> Dict[str, Any]:
        """
        Make final decision on action status using evidence-first rules
        
        Decision rules:
        1. If negation in transcript → absent (override prior)
        2. If transcript score ≥ 0.5 → present (transcript evidence)
        3. If prior score ≥ 0.4 and no negation → present (carry-over)
        4. Otherwise → unknown (insufficient evidence)
        
        Returns:
            Decision dict with status, source, scores, evidence
        """
        # Calculate scores
        s_t, strength, evidence_span = self.calculate_transcript_score(action, transcript)
        s_p = self.calculate_prior_score(action, previous_occurrences, category)
        
        decision = {
            "category": category,
            "action": action,
            "scores": {"s_t": round(s_t, 2), "s_p": round(s_p, 2)},
            "evidence": {
                "transcript_strength": strength,
                "transcript_span": evidence_span[:100] if evidence_span else "",
                "previous_visits": [visit_id for visit_id, _ in previous_occurrences]
            }
        }
        
        # Rule 1: Negation overrides everything
        if strength == "negation":
            decision.update({
                "status": "absent",
                "source": "transcript",
                "confidence": "high",
                "reason": "明確な否定表現あり"
            })
            return decision
        
        # Rule 2: Strong transcript evidence
        if s_t >= 0.5:
            decision.update({
                "status": "present",
                "source": "transcript" if s_p < 0.4 else "both",
                "confidence": "high" if s_t >= 0.8 else "medium",
                "reason": "会話内容で実施を確認"
            })
            return decision
        
        # Rule 3: Strong prior with no contradiction (threshold lowered to 0.4)
        if s_p >= 0.4:
            decision.update({
                "status": "present",
                "source": "previous",
                "confidence": "medium",
                "reason": "過去の記録から継続と推定（会話での否定なし）"
            })
            return decision
        
        # Rule 4: Insufficient evidence
        decision.update({
            "status": "unknown",
            "source": None,
            "confidence": "low",
            "reason": "十分な根拠なし"
        })
        return decision
    
    def extract_actions_from_previous_report(self, previous_report_text: str) -> Dict[str, List[str]]:
        """
        Extract actions from previous report text
        
        Returns:
            Dict mapping category to list of detected actions
        """
        detected_actions = {category: [] for category in self.canonical_actions.keys()}
        
        norm_text = self.normalize_text(previous_report_text)
        
        for category, actions in self.canonical_actions.items():
            for action in actions:
                norm_action = self.normalize_text(action)
                if norm_action in norm_text:
                    detected_actions[category].append(action)
        
        return detected_actions
    
    def process_all_actions(self, transcript: str, previous_reports: List[Dict[str, str]]) -> Dict[str, List[Dict[str, Any]]]:
        """
        Process all canonical actions against transcript and previous reports
        
        Args:
            transcript: Current visit transcript
            previous_reports: List of previous reports with 'text' and 'date' keys
        
        Returns:
            Dict mapping category to list of action decisions
        """
        # Build occurrence index from previous reports
        action_occurrences = {category: {action: [] for action in actions} 
                             for category, actions in self.canonical_actions.items()}
        
        for rank, prev_report in enumerate(previous_reports):
            prev_text = prev_report.get('text', '')
            visit_id = prev_report.get('date', f'visit_{rank}')
            
            detected = self.extract_actions_from_previous_report(prev_text)
            for category, detected_actions_list in detected.items():
                for detected_action in detected_actions_list:
                    if detected_action in action_occurrences[category]:
                        action_occurrences[category][detected_action].append((visit_id, rank))
        
        # Process each action
        results = {category: [] for category in self.canonical_actions.keys()}
        
        for category, actions in self.canonical_actions.items():
            for action in actions:
                decision = self.decide_action_status(
                    action, category, transcript, action_occurrences[category][action]
                )
                results[category].append(decision)
        
        return results
    
    def filter_present_actions(self, decisions: Dict[str, List[Dict[str, Any]]]) -> Dict[str, List[str]]:
        """
        Filter decisions to only include present actions, grouped by category
        
        Returns:
            Dict mapping category to list of present action names
        """
        present_actions = {}
        
        for category, action_list in decisions.items():
            present_in_category = [
                decision["action"] 
                for decision in action_list 
                if decision["status"] == "present"
            ]
            if present_in_category:
                present_actions[category] = present_in_category
        
        return present_actions
    
    def format_for_summary(self, present_actions: Dict[str, List[str]]) -> str:
        """
        Format present actions for inclusion in summary
        
        Returns:
            Formatted string suitable for 実施した看護・リハビリテーションの内容
        """
        formatted_lines = []
        
        for category in ["生活援助", "排泄援助", "処置", "リハビリ", "指導", "管理"]:
            if category in present_actions and present_actions[category]:
                actions_str = "　".join(present_actions[category])
                formatted_lines.append(f"- {category}: {actions_str}")
        
        return "\n".join(formatted_lines) if formatted_lines else ""
    
    def generate_decision_summary(self, decisions: Dict[str, List[Dict[str, Any]]]) -> Dict[str, Any]:
        """Generate summary statistics from decisions"""
        summary = {
            "total_actions": 0,
            "present_count": 0,
            "absent_count": 0,
            "unknown_count": 0,
            "by_source": {"transcript": 0, "previous": 0, "both": 0},
            "by_category": {}
        }
        
        for category, action_list in decisions.items():
            category_stats = {"present": 0, "absent": 0, "unknown": 0}
            
            for decision in action_list:
                summary["total_actions"] += 1
                status = decision["status"]
                
                if status == "present":
                    summary["present_count"] += 1
                    category_stats["present"] += 1
                    source = decision["source"]
                    if source in summary["by_source"]:
                        summary["by_source"][source] += 1
                elif status == "absent":
                    summary["absent_count"] += 1
                    category_stats["absent"] += 1
                else:
                    summary["unknown_count"] += 1
                    category_stats["unknown"] += 1
            
            summary["by_category"][category] = category_stats
        
        return summary

print("✅ Evidence-First Care Action System loaded!")
print("📋 Key features:")
print("   • Deterministic decision rules with explicit thresholds")
print("   • Negation detection (overrides priors)")
print("   • Provenance tracking (transcript vs. previous vs. both)")
print("   • Tunable persistence bias by category")
print("   • Full JSON output for transparency")
print("\n🎯 Decision Rules:")
print("   1. Negation in transcript → ABSENT (overrides prior)")
print("   2. Transcript score ≥ 0.5 → PRESENT (strong evidence)")
print("   3. Prior score ≥ 0.7 + no negation → PRESENT (carry-over)")
print("   4. Otherwise → UNKNOWN (insufficient evidence)")
print("\n💡 Usage:")
print("   system = EvidenceFirstCareActionSystem()")
print("   decisions = system.process_all_actions(transcript, previous_reports)")
print("   present_actions = system.filter_present_actions(decisions)")
print("   formatted_output = system.format_for_summary(present_actions)")

In [ ]:
# ============================================================================
# CELL 6.7: RAG INTEGRATION WITH EVIDENCE-FIRST SYSTEM
# ============================================================================

class EnhancedRAGWithEvidenceFirst(SimplifiedCompleteCrossPatientRAG):
    """
    Enhanced RAG system integrating Transcript-First Care Action System
    
    This combines your existing RAG pipeline with the transcript-first approach:
    1. Extract care actions from current transcript FIRST (ground truth)
    2. Supplement missing actions from previous reports
    3. Clear provenance tracking for each action
    """
    
    def __init__(self, api_key, prevreports_dir: Path = PREVREPORTS_DIR):
        super().__init__(api_key)
        self.evidence_system = TranscriptFirstCareActionSystem()
        self.prevreports_dir = prevreports_dir
    
    def extract_persistent_info_smartly(self, previous_report: str) -> str:
        """
        NEW SMART APPROACH: Extract stable facts + templates (not full text)
        
        This replaces the old AS-IS extraction to reduce hallucination
        
        Returns:
            Formatted string with stable facts and templates for prompt
        """
        # Use the smart extractor
        stable_facts = smart_extractor.extract_stable_facts(previous_report)
        templates = smart_extractor.generate_section_templates(previous_report)
        
        # Format for prompt
        formatted = smart_extractor.format_for_prompt(stable_facts, templates)
        
        return formatted
    
    # Keep old method for backward compatibility but deprecate it
    def extract_persistent_sections_from_previous_report(self, previous_report: str) -> Dict[str, str]:
        """
        DEPRECATED: Old method that extracted full sections AS-IS
        Use extract_persistent_info_smartly() instead
        
        This is kept only for backward compatibility
        """
        print("⚠️ Warning: Using deprecated method. Consider using extract_persistent_info_smartly()")
        
        persistent_sections = {}
        
        # Define target sections
        section_names = [
            '<処置内容・訓練内容>',
            '<備考>',
            '<外部連絡・伝達事項>',
            '<内部連絡・伝達事項>'
        ]
        
        for section_name in section_names:
            # Extract section content - match until next section tag or end
            pattern = rf'{re.escape(section_name)}(.*?)(?=<[^>]+>|$)'
            match = re.search(pattern, previous_report, re.DOTALL)
            
            if match:
                content = match.group(1).strip()
                if content:  # Only store non-empty sections
                    persistent_sections[section_name] = content
        
        return persistent_sections
    
    def format_persistent_sections_for_prompt(self, persistent_sections: Dict[str, str]) -> str:
        """
        DEPRECATED: Format extracted persistent sections for LLM prompt
        This method is kept for backward compatibility
        """
        if not persistent_sections:
            return ""
        
        formatted_parts = []
        
        # Format in order of importance
        section_order = [
            '<備考>',
            '<処置内容・訓練内容>',
            '<内部連絡・伝達事項>',
            '<外部連絡・伝達事項>'
        ]
        
        for section_name in section_order:
            if section_name in persistent_sections:
                content = persistent_sections[section_name]
                formatted_parts.append(f"{section_name}\n{content}")
        
        return "\n\n".join(formatted_parts)
    
    def load_previous_report_data(self, patient_id: str) -> Optional[Dict[str, str]]:
        """
        Load previous report data for evidence-first processing + persistent facts
        
        Returns:
            Dict with 'text' containing the full previous report with <備考> section
        """
        patient_subfolder = self.prevreports_dir / patient_id
        if not patient_subfolder.exists():
            return None
        
        prev_file = patient_subfolder / f"PrevReports - {patient_id}.csv"
        if not prev_file.exists():
            return None
        
        try:
            df = pd.read_csv(prev_file)
            if df.empty or 'Field' not in df.columns or 'Value' not in df.columns:
                return None
            
            field_value_dict = dict(zip(df['Field'], df['Value']))
            
            # ✅ Use '利用者の状態（病状・トピックス）' field which contains <備考> with persistent facts
            user_status = str(field_value_dict.get('利用者の状態（病状・トピックス）', '')).strip()
            nursing_content = str(field_value_dict.get('実施した看護・リハビリテーションの内容', '')).strip()
            transcription = str(field_value_dict.get('Transcription', '')).strip()
            date = str(field_value_dict.get('Date', 'Unknown')).strip()
            
            # Use user_status as primary source (contains <備考> section)
            if not user_status and not nursing_content:
                return None
            
            # Combine for full context - prioritize user_status which has <備考>
            full_text = ""
            if user_status:
                full_text += f"利用者の状態（病状・トピックス）:\n{user_status}\n\n"
            if nursing_content:
                full_text += f"実施した看護・リハビリテーションの内容:\n{nursing_content}\n\n"
            if transcription:
                full_text += f"会話記録:\n{transcription}"
            
            return {
                'text': full_text,
                'user_status': user_status,
                'nursing_content': nursing_content,
                'date': date
            }
            
        except Exception as e:
            print(f"❌ Error loading previous report for {patient_id}: {e}")
            return None
    
    async def generate_evidence_enhanced_summary(
        self,
        transcript: str,
        patient_id: str,
        categories: List[str],
        patient_history: List[Dict]
    ) -> Tuple[str, Dict]:
        """
        Generate summary using Transcript-First system + RAG
        
        This method:
        1. Extracts care actions from transcript FIRST (ground truth)
        2. Supplements missing actions from previous reports
        3. Uses RAG for other contextual information
        4. Combines all for complete, accurate summary with clear provenance
        """
        
        print(f"\n🔬 Transcript-First Enhanced Processing for {patient_id}")
        print("="*60)
        
        # Load previous report for Transcript-First processing
        prev_report = self.load_previous_report_data(patient_id)
        
        # Process care actions with Transcript-First system
        care_action_decisions = None
        care_action_summary = None
        persistent_facts = {}
        persistent_facts_str = ""
        
        if prev_report:
            print("✅ Previous report found - applying Transcript-First analysis")
            previous_reports = [prev_report]
            
            # Get action decisions (transcript first, then supplement)
            # Returns: (transcript_decisions, supplementary_decisions, merged_decisions)
            transcript_acts, supplement_acts, merged_acts = self.evidence_system.process_transcript_first(
                transcript, previous_reports
            )
            
            # Use merged_decisions for final output
            care_action_decisions = merged_acts
            
            # Generate statistics from merged decisions
            stats = self.evidence_system.generate_decision_summary(care_action_decisions)
            print(f"   • Total actions: {stats['total_actions']}")
            print(f"   • Present: {stats['present_count']} (transcript: {stats['by_source']['transcript']}, "
                  f"supplemented: {stats['by_source']['previous']}, both: {stats['by_source']['both']})")
            
            # Format care actions for summary
            present_actions = self.evidence_system.filter_present_actions(care_action_decisions)
            care_action_summary = self.evidence_system.format_for_summary(present_actions)
            
            # ✅ NEW: EXTRACT SMART PERSISTENT INFO (stable facts + templates)
            print("\n🔍 Extracting smart persistent info from previous report...")
            persistent_facts_str = self.extract_persistent_info_smartly(prev_report['text'])
            if persistent_facts_str:
                # Count extracted facts
                fact_count = persistent_facts_str.count('•')
                print(f"   ✅ Extracted {fact_count} stable facts + section templates")
                print("   ℹ️  Using hybrid approach: stable facts + structure guidance")
            else:
                print("   ⚠️ No persistent information extracted")
        else:
            print("⚠️ No previous report found - using standard RAG only")
        
        # Use standard RAG for other sections (categories, context)
        similar_patients = self.simplified_system.find_same_disease_patients_only(patient_id, top_k=3)
        
        strategy, strategy_explanation = self.strategy_selector.select_strategy(
            len(patient_history), similar_patients
        )
        
        print(f"\n✨ RAG Strategy: {strategy}")
        print(f"📊 {strategy_explanation}")
        
        # Prepare RAG context
        cross_patient_context = None
        earliest_history = None
        
        if strategy in ['hybrid', 'cross_patient_only']:
            cross_patient_context = await self.build_cross_patient_context(transcript, similar_patients)
        
        if strategy in ['hybrid', 'history_only']:
            earliest_history = patient_history[0] if patient_history else None
        
        # Generate complete summary with Transcript-First integration + Persistent Facts
        summary = await self.generate_unified_summary_with_evidence(
            transcript=transcript,
            categories=categories,
            care_action_summary=care_action_summary,
            care_action_decisions=care_action_decisions,
            persistent_facts_str=persistent_facts_str,
            earliest_history=earliest_history,
            cross_patient_context=cross_patient_context
        )
        
        # Compile strategy data
        strategy_data = {
            'strategy_used': strategy,
            'strategy_explanation': strategy_explanation,
            'similar_patients': [p['patient_id'] for p in similar_patients],
            'transcript_first_enabled': care_action_decisions is not None,
            'care_action_stats': stats if care_action_decisions else None,
            'safety_design': 'transcript_first_with_rag'
        }
        
        return summary, strategy_data
    
    async def generate_unified_summary_with_evidence(
        self,
        transcript: str,
        categories: List[str],
        care_action_summary: Optional[str],
        care_action_decisions: Optional[Dict],
        persistent_facts_str: str = "",
        earliest_history: Optional[Dict] = None,
        cross_patient_context: Optional[List[Dict]] = None
    ) -> str:
        """
        Enhanced summary generation with Transcript-First care actions + Persistent Facts
        SINGLE-PASS approach with anti-abstraction prompts
        """
        
        # Build context sections
        earliest_context_str = ""
        if earliest_history:
            history_text_cleaned = strip_numeric_data(earliest_history.get('text', ''))
            date_info = earliest_history.get('date', 'Unknown')
            date_str = f"({date_info})" if date_info and date_info != 'Unknown' else ""
            earliest_context_str = f"""
== 患者の前回の診察記録 {date_str} ==
{history_text_cleaned}"""
        
        cross_patient_context_str = ""
        if cross_patient_context:
            cross_patient_context_str = "\n== 同一疾患患者の参考事例（参考コンテキストのみ）==\n"
            cross_patient_context_str += "【禁止事項】以下の事例から新規項目を追加することは禁止です。表現の参考のみに使用してください。\n\n"
            for i, ctx in enumerate(cross_patient_context[:3], 1):
                cross_patient_context_str += f"""
事例{i} (患者{ctx['source_patient']}, 関連度: {ctx['relevance_score']:.2f})
{ctx['context_text']}
---"""
        
        # Transcript-First context (if available)
        evidence_context_str = ""
        if care_action_summary and care_action_decisions:
            # Build detailed evidence section
            evidence_items = []
            for category, decisions in care_action_decisions.items():
                present_items = [d for d in decisions if d['status'] == 'present']
                if present_items:
                    transcript_items = [d['action'] for d in present_items if d['source'] == 'transcript']
                    previous_items = [d['action'] for d in present_items if d['source'] == 'previous']
                    
                    if transcript_items:
                        evidence_items.append(f"  • {category} (会話で確認): {', '.join(transcript_items)}")
                    if previous_items:
                        evidence_items.append(f"  • {category} (前回から補完): {', '.join(previous_items)}")
            
            evidence_context_str = f"""
== 看護・リハビリ実施内容（Transcript-First解析済み） ==

{chr(10).join(evidence_items)}

【重要】上記は会話優先の決定論的ルールで判定済みです：
• 会話で明確に言及された項目（最優先）
• 前回記録から補完された項目（会話で言及なし＋否定なし）
• 推測や創作は含まれていません

このリストを「実施した看護・リハビリテーションの内容」セクションに使用してください。
"""
        
        # Strategy rules
        has_evidence = bool(care_action_summary)
        has_history = bool(earliest_history)
        has_cross = bool(cross_patient_context)
        has_persistent = bool(persistent_facts_str)
        
        if has_evidence:
            strategy_rules = """【最優先指示 - Transcript-First決定テーブル】
• 「実施した看護・リハビリテーションの内容」は会話優先の決定論的ルールで判定済みです
• このリストに含まれる項目のみを記載してください
• 追加・削除・推測は一切禁止です
• 参考事例や過去記録から新規項目を追加しないでください

【新しいアプローチ - 継続管理情報】
以下のセクションは「スマート抽出」により、安定した情報のみ参考として提供されています：
• <備考>: 透析、ストーマ、服薬などの継続管理情報（変更頻度の低い情報のみ）
• <処置内容・訓練内容>: 定期処置の基本情報
• <外部連絡・伝達事項>: 外部機関連絡（必要時のみ）
• <内部連絡・伝達事項>: スタッフ申し送り（必要時のみ）

【重要】これらは単純コピーせず、今回の会話内容を最優先して適切に更新してください。
"""
        elif has_history and has_cross:
            strategy_rules = """【情報源の優先順位】
1. 最優先: 現在の訪問記録（会話内容）
2. 参考: 本患者の前回記録から抽出された安定情報（スマート抽出）
3. 参考: 同一疾患患者の経験（処置方法や観察ポイントの参考程度）

【新しいアプローチ】
継続管理情報は「スマート抽出」方式を採用：
• 変更頻度の低い事実データのみ抽出（全文コピーなし）
• セクション構成テンプレート提供
• 今回の会話内容を最優先して更新
"""
        else:
            strategy_rules = """【特記事項】
• 会話内容を最優先に使用
• 推測や創作は禁止
• 継続管理情報は「スマート抽出」により安定情報のみ参考として提供
• 単純コピーではなく、会話に基づいて適切に更新すること"""
        
        # Persistent facts context (if available) - NEW SMART APPROACH
        persistent_context_str = ""
        if has_persistent:
            persistent_context_str = f"""
== 前回記録からの継続管理情報（スマート抽出） ==

{persistent_facts_str}

【新しいアプローチの説明】
上記は前回の訪問記録から、変更頻度の低い安定した情報のみを抽出しました。
• 抽出内容：透析スケジュール、装具仕様、服薬パターンなどの構造化データ
• 推奨構成：<備考>等のセクション構成テンプレート

【重要な使用指示】
✅ このアプローチの利点：
• 推測や創作のリスクを大幅削減
• LLMが今回の会話に基づいて適切に更新可能
• 安定情報と変動情報を明確に区別

📋 セクション作成時：
1. 今回の会話で明確に述べられた内容を最優先
2. 会話で言及がない項目は、上記の参考情報を利用
3. 会話で「変更なし」と述べられた項目は、上記を使用
4. 詳細が不要な場合は「前回記録参照」と簡潔に記載可

⚠️ 禁止事項：
• 上記の情報を単純にコピーペーストしないこと
• 推測や創作で情報を追加しないこと
• 古い情報を無条件に引き継がないこと
"""
        
        # Build prompt
        enhanced_categories = inject_topics_category(categories, transcript)
        cats_lines = "\n".join(f"【{c}】" for c in enhanced_categories)
        
        # Build unified user prompt (SINGLE-PASS, with persistent facts)
        user_prompt = f"""
{strategy_rules}

【現在の訪問記録】
{transcript}

{evidence_context_str}

{persistent_context_str}

{earliest_context_str}

{cross_patient_context_str}

動的カテゴリ -（以下の順序で必ず全て記載）
{cats_lines}

【重要】
• 【利用者の状態（病状・トピックス）】セクションでは、具体的な記載を心がけてください
• 会話に出た具体的な薬剤名、食べ物、場所、患者の発言を必ず含めてください
• 抽象的表現（不規則、不安定、不十分、良好など）は使用禁止
• 4つの継続管理セクション（<処置内容・訓練内容>、<備考>、<外部連絡・伝達事項>、<内部連絡・伝達事項>）は、前回記録から引き継いでください
"""
        
        # API call
        try:
            encoding = tiktoken.encoding_for_model(MODEL_SUMMARY)
        except Exception:
            encoding = tiktoken.get_encoding("o200k_base")
        
        system_tokens = len(encoding.encode(self.system_message))
        user_tokens = len(encoding.encode(user_prompt))
        input_tokens = system_tokens + user_tokens
        
        response = await self.client.chat.completions.create(
            model=MODEL_SUMMARY,
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMP_SUMMARY,
        )
        
        output_tokens = response.usage.completion_tokens if response.usage else 0
        actual_input_tokens = response.usage.prompt_tokens if response.usage else input_tokens
        processing_meter.track_api_call('summary', MODEL_SUMMARY, actual_input_tokens, output_tokens)
        
        return response.choices[0].message.content

print("✅ Enhanced RAG with Transcript-First System loaded!")
print("📋 Class available:")
print("   • EnhancedRAGWithEvidenceFirst - Integrated RAG + Transcript-First")
print("\n🎯 Features:")
print("   • Transcript-first care action detection (3-step flow)")
print("   • Extract from current visit FIRST, then supplement from previous")
print("   • Clear provenance tracking (transcript vs supplemented)")
print("   • Seamless integration with existing RAG")
print("   • Backward compatible with standard pipeline")
print("\n💡 Usage:")
print("   rag = EnhancedRAGWithEvidenceFirst(api_key=OPENAI_API_KEY)")
print("   summary, strategy = await rag.generate_evidence_enhanced_summary(")
print("       transcript, patient_id, categories, patient_history)")

In [ ]:
# ============================================================================
# ENHANCEMENTS TO EVIDENCE-FIRST SYSTEM
# Expanded markers and confidence calibration for TranscriptFirstCareActionSystem
# ============================================================================

# Expanded execution verbs for better transcript detection
EXECUTION_VERBS_EXPANDED = [
    # Original
    "実施", "行った", "施行", "塗布", "交換", "介助", "訓練", "指導", "管理", "貼付",
    # Additional nursing-specific verbs
    "確認", "観察", "測定", "チェック", "実施した", "行いました", "しました",
    "変更", "調整", "継続", "開始", "実施中", "施行中",
    "貼り替え", "張り替え", "取り替え", "付け替え",
    "拭いた", "洗った", "清拭", "消毒",
    "声かけ", "傾聴", "相談", "説明",
]

# Expanded negation markers
NEGATION_MARKERS_EXPANDED = [
    # Original
    "中止", "不要", "今回はなし", "未実施", "見送り", "保留", "実施なし", "行わず",
    # Additional patterns
    "今日はなし", "本日なし", "今回なし", "今日は不要", "本日不要",
    "中断", "やめた", "止めた", "辞めた",
    "できなかった", "できない", "しなかった", "していない",
    "拒否", "希望なし", "必要なし",
    "該当なし", "対象外",
]

def calculate_confidence_score(s_t: float, s_p: float, strength: str, source: str) -> Tuple[str, float]:
    """
    Calculate calibrated confidence score
    
    Returns:
        (confidence_label, confidence_score_0_to_1)
    """
    # Very high confidence - strong transcript evidence
    if strength == "strong" and s_t >= 0.9:
        return "very_high", 0.95
    
    # High confidence - clear evidence
    elif s_t >= 0.7 or (s_t >= 0.5 and s_p >= 0.6):
        return "high", 0.85
    
    # Medium-high - good transcript or strong prior
    elif s_t >= 0.5 or s_p >= 0.7:
        return "medium_high", 0.75
    
    # Medium - borderline cases
    elif (0.4 <= s_p < 0.7 and s_t < 0.5) or (0.3 <= s_t < 0.5):
        return "medium", 0.60
    
    # Low - weak evidence (flagged for review)
    elif 0.3 <= s_p < 0.4 or 0.2 <= s_t < 0.3:
        return "low", 0.45
    
    # Very low / uncertain
    else:
        return "very_low", 0.25

print("✅ Evidence-First Enhancements Loaded!")
print("📋 Active features:")
print("   • Expanded execution verbs (確認, 観察, チェック, etc.)")
print("   • Expanded negation markers (今日はなし, 中断, etc.)")
print("   • Confidence calibration (5 levels: very_high → very_low)")
print("\n💡 Usage:")
print("   conf_label, conf_score = calculate_confidence_score(s_t, s_p, strength, source)")

In [ ]:
# ============================================================================
# TRANSCRIPT-FIRST SYSTEM (What You Actually Want!)
# Extract from transcript FIRST, then supplement from previous report
# ============================================================================

class TranscriptFirstCareActionSystem(EvidenceFirstCareActionSystem):
    """
    Transcript-first approach:
    1. Extract all actions mentioned in current transcript (PRIMARY)
    2. Check previous report for routine actions not mentioned (SUPPLEMENTARY)
    3. Merge both lists with clear provenance
    
    Philosophy:
    - Current transcript = ground truth for THIS visit
    - Previous report = source for routine actions not explicitly discussed
    - If transcript is silent → check if it was routine in previous → carry over
    """
    
    def __init__(self):
        super().__init__()
        self.execution_verbs = EXECUTION_VERBS_EXPANDED
        self.negation_markers = NEGATION_MARKERS_EXPANDED
        self.prior_threshold = 0.4  # For carry-over decisions
    
    def has_execution_verb(self, context: str) -> bool:
        return any(verb in context for verb in self.execution_verbs)
    
    def has_negation(self, context: str) -> bool:
        return any(marker in context for marker in self.negation_markers)
    
    def extract_from_transcript(self, transcript: str) -> Dict[str, List[Dict]]:
        """
        STEP 1: Extract all actions from current transcript
        
        Returns:
            Dict mapping category to list of {action, status, confidence, evidence}
        """
        transcript_actions = {category: [] for category in self.canonical_actions.keys()}
        
        for category, actions in self.canonical_actions.items():
            for action in actions:
                s_t, strength, evidence_span = self.calculate_transcript_score(action, transcript)
                
                # If action is mentioned in transcript (positive or negative)
                if s_t > 0 or strength == "negation":
                    if strength == "negation":
                        status = "absent"
                        conf = "high"
                        reason = "会話で明確に否定"
                    elif s_t >= 0.7:
                        status = "present"
                        conf = "high"
                        reason = "会話で明確に実施確認"
                    elif s_t >= 0.5:
                        status = "present"
                        conf = "medium"
                        reason = "会話で実施言及"
                    else:
                        status = "present"
                        conf = "low"
                        reason = "会話で弱い言及"
                    
                    transcript_actions[category].append({
                        'action': action,
                        'status': status,
                        'source': 'transcript',
                        'confidence': conf,
                        's_t': round(s_t, 2),
                        'evidence': evidence_span[:100] if evidence_span else "",
                        'reason': reason
                    })
        
        return transcript_actions
    
    def supplement_from_previous(self, transcript: str, previous_reports: List[Dict[str, str]], 
                                 transcript_actions: Dict[str, List[Dict]]) -> Dict[str, List[Dict]]:
        """
        STEP 2: For actions NOT in transcript, check if they should carry over from previous
        
        Args:
            transcript: Current transcript
            previous_reports: Previous visit reports
            transcript_actions: Actions already extracted from transcript
        
        Returns:
            Supplementary actions from previous report
        """
        # Get actions already handled by transcript
        transcript_action_names = {category: set(a['action'] for a in actions) 
                                   for category, actions in transcript_actions.items()}
        
        # Build occurrence index from previous reports
        action_occurrences = {category: {action: [] for action in actions} 
                             for category, actions in self.canonical_actions.items()}
        
        for rank, prev_report in enumerate(previous_reports):
            prev_text = prev_report.get('text', '')
            visit_id = prev_report.get('date', f'visit_{rank}')
            
            detected = self.extract_actions_from_previous_report(prev_text)
            for category, detected_actions_list in detected.items():
                for detected_action in detected_actions_list:
                    if detected_action in action_occurrences[category]:
                        action_occurrences[category][detected_action].append((visit_id, rank))
        
        # For each action NOT in transcript, decide if it should carry over
        supplementary_actions = {category: [] for category in self.canonical_actions.keys()}
        
        for category, actions in self.canonical_actions.items():
            for action in actions:
                # Skip if already handled by transcript
                if action in transcript_action_names[category]:
                    continue
                
                # Check if it was in previous reports
                prev_occurrences = action_occurrences[category][action]
                if not prev_occurrences:
                    continue  # Not in previous either, so unknown
                
                # Calculate prior score for carry-over decision
                s_p = self.calculate_prior_score(action, prev_occurrences, category)
                
                # Decide if it should carry over (since transcript is silent)
                if s_p >= self.prior_threshold:
                    conf_label, conf_score = calculate_confidence_score(0, s_p, "none", "previous")
                    supplementary_actions[category].append({
                        'action': action,
                        'status': 'present',
                        'source': 'previous',
                        'confidence': conf_label,
                        's_p': round(s_p, 2),
                        'evidence': f"前回: {', '.join(v for v,_ in prev_occurrences)}",
                        'reason': f"会話で言及なし、前回から継続と推定 (s_p={s_p:.2f})"
                    })
        
        return supplementary_actions
    
    def process_transcript_first(self, transcript: str, 
                                previous_reports: List[Dict[str, str]]) -> Tuple[Dict, Dict, Dict]:
        """
        Main processing: Transcript first, then supplement from previous
        
        Returns:
            (transcript_decisions, supplementary_decisions, merged_decisions)
        """
        # STEP 1: Extract from current transcript
        transcript_decisions = self.extract_from_transcript(transcript)
        
        # STEP 2: Supplement from previous report (for actions not in transcript)
        supplementary_decisions = self.supplement_from_previous(
            transcript, previous_reports, transcript_decisions
        )
        
        # STEP 3: Merge both
        merged_decisions = {category: [] for category in self.canonical_actions.keys()}
        for category in self.canonical_actions.keys():
            merged_decisions[category] = (
                transcript_decisions[category] + 
                supplementary_decisions[category]
            )
        
        return transcript_decisions, supplementary_decisions, merged_decisions
    
    def generate_summary_report(self, transcript_decisions: Dict, 
                               supplementary_decisions: Dict, 
                               merged_decisions: Dict) -> Dict:
        """
        Generate comprehensive summary with clear breakdown
        """
        # Count actions from each source
        transcript_count = sum(len(actions) for actions in transcript_decisions.values())
        transcript_present = sum(1 for cat in transcript_decisions.values() 
                                for a in cat if a['status'] == 'present')
        transcript_absent = sum(1 for cat in transcript_decisions.values() 
                               for a in cat if a['status'] == 'absent')
        
        supplement_count = sum(len(actions) for actions in supplementary_decisions.values())
        
        merged_present = sum(1 for cat in merged_decisions.values() 
                           for a in cat if a['status'] == 'present')
        
        return {
            'transcript_summary': {
                'mentioned': transcript_count,
                'present': transcript_present,
                'absent': transcript_absent
            },
            'supplement_summary': {
                'carried_over': supplement_count
            },
            'merged_summary': {
                'total_present': merged_present,
                'from_transcript': transcript_present,
                'from_previous': supplement_count
            }
        }


print("✅ Transcript-First Care Action System Loaded!")
print("=" * 70)
print("🎯 Flow:")
print("   1️⃣  Extract from CURRENT transcript (primary source)")
print("   2️⃣  Supplement from previous report (routine actions not mentioned)")
print("   3️⃣  Merge both with clear provenance")
print("\n📋 Philosophy:")
print("   • Transcript = ground truth for THIS visit")
print("   • Previous = source for routine care not explicitly discussed")
print("   • If silent in transcript + was routine → carry over")
print("\n💡 Usage:")
print("   system = TranscriptFirstCareActionSystem()")
print("   transcript_acts, supplement_acts, merged = system.process_transcript_first(transcript, prev_reports)")
print("   summary = system.generate_summary_report(transcript_acts, supplement_acts, merged)")
print("=" * 70)

In [ ]:
# ============================================================================
# CELL 7: PIPELINE ORCHESTRATION
# ============================================================================

# =========================
# Excel Form Processing Functions
# =========================

def load_form_sheets(form_path: Path) -> Dict[str, pd.DataFrame]:
    """Load all sheets from an Excel form file"""
    try:
        sheets = pd.read_excel(form_path, sheet_name=None, header=None)
        print(f"✅ Loaded {len(sheets)} sheets from {form_path.name}")
        return sheets
    except Exception as e:
        print(f"❌ Error loading Excel file {form_path}: {e}")
        return {}

def extract_category_names(sheet_df: pd.DataFrame,
                          section_header: str = "利用者の状態（病状・トピックス）") -> List[str]:
    """Extract category names from form sheet"""
    if sheet_df is None or sheet_df.empty:
        return []

    # Locate section row
    idxs = sheet_df[
        sheet_df.apply(lambda row: row.astype(str).str.contains(section_header, na=False).any(), axis=1)
    ].index

    if len(idxs) == 0:
        print(f"⚠️ Section header '{section_header}' not found in sheet")
        return []

    start = idxs[0]
    cat_pat = r"【(.*?)】"
    cats: List[str] = []

    # Look for categories below the header
    for r in range(start + 1, sheet_df.shape[0]):
        row = sheet_df.iloc[r]
        if row.isna().all():
            break
        text = " ".join(row.dropna().astype(str))
        cats.extend(re.findall(cat_pat, text))

    # Normalize & dedupe
    seen = set()
    out = []
    for c in cats:
        c = c.strip()
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    
    print(f"🎯 Extracted {len(out)} categories: {out}")
    return out

def find_visiting_form_excels(root: Path) -> List[Path]:
    """Find all non-temp Excel files under root"""
    if not root.exists():
        print(f"❌ Root directory does not exist: {root}")
        return []
    
    excel_files = []
    for p in root.rglob("*"):
        if (p.is_file() and 
            p.suffix.lower() in {".xlsx", ".xls"} and 
            not p.name.startswith("~$")):
            excel_files.append(p)
    
    excel_files.sort()
    print(f"📋 Found {len(excel_files)} Excel files in {root}")
    return excel_files

def map_form_to_transcript_path(form_path: Path, form_root: Path, transcript_root: Path) -> Path:
    """Map a form Excel file to its corresponding transcript file by matching filenames"""
    
    relative_path = form_path.relative_to(form_root)
    patient_folder = relative_path.parts[0] if relative_path.parts else ""
    
    # Extract patient ID
    patient_id = ""
    if patient_folder.startswith('P') and patient_folder[1:].isdigit():
        patient_id = patient_folder
    else:
        match = re.search(r'P(\d+)', form_path.name)
        if match:
            patient_id = f"P{match.group(1)}"
    
    if not patient_id:
        print(f"⚠️ Could not extract patient ID from {form_path}")
        return transcript_root / "unknown.txt"
    
    # Extract the base filename from form (without extension)
    # e.g., "audio_example_visit_0001.xlsx" -> "audio_example_visit_0001"
    form_stem = form_path.stem
    
    # Look for transcript with matching filename
    transcript_patient_dir = transcript_root / patient_id
    if transcript_patient_dir.exists():
        # Try to find exact match first
        matching_transcript = transcript_patient_dir / f"{form_stem}.txt"
        if matching_transcript.exists():
            print(f"🎯 Exact match: {form_path.name} -> {matching_transcript.name}")
            return matching_transcript
        
        # Fallback: look for any txt file with similar name pattern
        txt_files = list(transcript_patient_dir.glob("*.txt"))
        for txt_file in txt_files:
            if txt_file.stem == form_stem:
                print(f"🎯 Matched: {form_path.name} -> {txt_file.name}")
                return txt_file
        
        # If no exact match found, warn but return first file (backward compatibility)
        if txt_files:
            print(f"⚠️ No exact match for {form_path.name}, using {txt_files[0].name}")
            return txt_files[0]
    
    print(f"⚠️ No transcript found for patient {patient_id}")
    return transcript_root / patient_id / "not_found.txt"

# =========================
# Single Form Processing
# =========================

async def run_one_form_optimized(
    form_path: Path,
    form_root: Path,
    transcript_root: Path,
    rag_system: EnhancedRAGWithEvidenceFirst,
    prevreports_dir: Path = PREVREPORTS_DIR
) -> Dict[str, Any]:
    """Optimized form processing with category extraction and async pipeline"""
    
    try:
        print(f"🏥 Processing form: {form_path.name}")
        
        # 1) Extract categories from form
        sheets = load_form_sheets(form_path)
        sheet1 = sheets.get("Sheet1")
        if sheet1 is None and len(sheets) > 0:
            sheet1 = list(sheets.values())[0]
        
        cats = extract_category_names(sheet1, "利用者の状態（病状・トピックス）")
        
        if not cats:
            cats = ["睡眠", "疼痛", "活動", "服薬", "水分", "栄養"]
            print(f"⚠️ No categories found, using fallback: {cats}")

        # 2) Find corresponding transcript
        tpath = map_form_to_transcript_path(form_path, form_root, transcript_root)
        if not tpath.exists():
            return {
                'status': 'failed',
                'form_path': str(form_path),
                'error': f"Transcript not found: {tpath}",
                'categories_extracted': cats
            }

        # 3) Extract patient ID
        patient_id = tpath.parent.name if tpath.parent.name.startswith('P') else 'Unknown'
        
        # 4) Read transcript
        transcript_text = read_transcript_text(tpath)
        if not transcript_text:
            return {
                'status': 'failed',
                'form_path': str(form_path),
                'transcript_path': str(tpath),
                'error': 'Empty or unreadable transcript',
                'categories_extracted': cats
            }

        # 5) Load patient history
        patient_history = load_patient_history(patient_id, prevreports_dir)

        # 6) Process with RAG system
        # Extraction
        extraction = await extract_with_json_mode(transcript_text, patient_id)
        
        # ✅ USE IMPROVED CROSS-PATIENT ENHANCED SUMMARY GENERATION
        summary, strategy_data = await rag_system.generate_evidence_enhanced_summary(
            transcript=transcript_text,
            patient_id=patient_id,
            categories=cats,
            patient_history=patient_history
        )

        # 7) Save results
        extraction_file = await save_extraction_results(extraction, patient_id, str(tpath))
        
        summaries_dir = OUT_DIR / "complete_pipeline_results" / "summaries"
        summaries_dir.mkdir(parents=True, exist_ok=True)
        summary_filename = f"form_summary_{patient_id}_{form_path.stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        summary_path = summaries_dir / summary_filename
        summary_path.write_text(summary, encoding='utf-8')

        return {
            'status': 'success',
            'patient_id': patient_id,
            'form_path': str(form_path),
            'transcript_path': str(tpath),
            'categories_extracted': cats,
            'extraction_file': extraction_file,
            'summary_file': str(summary_path),
            'strategy': strategy_data,
            'processing_type': 'improved_cross_patient_with_dynamic_categories'
        }

    except Exception as e:
        return {
            'status': 'failed',
            'form_path': str(form_path),
            'error': str(e),
            'categories_extracted': cats if 'cats' in locals() else []
        }

# =========================
# Batch Processing Pipeline
# =========================

_semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

async def _with_semaphore(coro):
    async with _semaphore:
        return await coro

async def process_all_forms_complete_pipeline(limit: Optional[int] = 10):
    """
    Process Excel form files with dynamic category extraction
    
    Args:
        limit: Process only first N files (None for all)
    """
    print("🚀 Starting form-based pipeline...")
    print(f"⚙️  Processing Excel forms with dynamic category extraction")
    
    processing_meter.start_timer()

    # Find all Excel form files
    if not FORM_ROOT.exists():
        print(f"❌ Form root directory not found: {FORM_ROOT}")
        return {"error": "Form root directory not found", "total_files": 0, "processed": 0}

    all_form_files = find_visiting_form_excels(FORM_ROOT)
    
    if limit is not None:
        all_form_files = all_form_files[:limit]

    total_files = len(all_form_files)
    print(f"📊 Found {total_files} Excel form files to process")

    if total_files == 0:
        print("⚠️ No Excel files found in the form directory")
        return {"total_files": 0, "processed": 0}

    # Load all patient data and initialize RAG
    all_patient_data = await load_all_patient_data()
    
    # ✅ USE ENHANCED RAG WITH EVIDENCE-FIRST SYSTEM
    rag = EnhancedRAGWithEvidenceFirst(api_key=OPENAI_API_KEY)
    await rag.initialize_cross_patient_knowledge(all_patient_data)
    
    print("✅ Using EnhancedRAGWithEvidenceFirst system")
    print("   • Transcript-First evidence processing")
    print("   • Smart persistent fact extraction")
    print("   • Improved three-stage semantic filtering")
    print("   • Adaptive threshold based on history quality")
    print("   • Weighted cross-patient integration (20-40%)")

    # Create tasks with semaphore
    tasks = [
        _with_semaphore(run_one_form_optimized(
            form_path=form_path,
            form_root=FORM_ROOT,
            transcript_root=TRANSCRIPT_ROOT,
            rag_system=rag,
            prevreports_dir=PREVREPORTS_DIR
        ))
        for form_path in all_form_files
    ]

    results = []
    # Run in bounded parallel batches
    BATCH = MAX_CONCURRENCY * 2
    for i in range(0, len(tasks), BATCH):
        batch = tasks[i:i+BATCH]
        batch_results = await asyncio.gather(*batch, return_exceptions=True)
        for r in batch_results:
            if isinstance(r, Exception):
                print(f"⚠️ Form processing task failed: {r}")
            else:
                results.append(r)
        # Persist caches between batches
        persist_all_caches()

    # Save batch index
    results_dir = OUT_DIR / "complete_pipeline_results"
    results_dir.mkdir(parents=True, exist_ok=True)
    idx_path = results_dir / f"form_batch_index_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    idx_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')

    print(f"✅ Completed processing {len(results)}/{total_files} form files")
    print(f"📄 Index saved: {idx_path}")

    # Final cache persist
    persist_all_caches()
    
    processing_meter.end_timer()
    processing_meter.print_summary()

    successful_results = [r for r in results if r.get('status') == 'success']
    failed_results = [r for r in results if r.get('status') == 'failed']

    print(f"\n📊 Final Results:")
    print(f"   ✅ Success: {len(successful_results)}")
    print(f"   ❌ Failed: {len(failed_results)}")

    return {
        "total_files": total_files,
        "processed": len(results),
        "successful": len(successful_results),
        "failed": len(failed_results),
        "index_file": str(idx_path),
        "sample": successful_results[:3],
    }

print("✅ Pipeline orchestration loaded!")
print("🚀 Main pipeline function:")
print("   • await process_all_forms_complete_pipeline(limit=10)")
print("📋 Features:")
print("   • ✨ IMPROVED cross-patient RAG with smart context cleaning")
print("   • Dynamic category extraction from Excel forms")
print("   • Quality-based strategy selection")
print("   • Structured pattern extraction")
print("   • Concurrent processing with rate limiting")
print("   • Comprehensive metering and cost tracking")
print("   • Automatic caching and persistence")
print("\n💡 Quick start:")
print("   results = await process_all_forms_complete_pipeline(limit=5)")
print("   # Process first 5 forms with ImprovedCompleteCrossPatientRAG pipeline")


In [ ]:
# ============================================================================
# TEST: Run Evidence-First Pipeline with limit=5
# ============================================================================

# Process first 5 forms with the new Evidence-First system
results = await process_all_forms_complete_pipeline(limit=None)

In [ ]:
# ============================================================================
# EXPERIMENTAL COMPARISON FRAMEWORK
# ============================================================================

import pandas as pd
from pathlib import Path
from typing import Dict, List
import asyncio

class ExperimentalFramework:
    """
    Systematic framework for comparing HK-RAG variants
    
    Tests:
    1. Original (threshold=0.5, no weighted prompts)
    2. Adaptive Threshold Only (threshold=0.65 for hybrid)
    3. Weighted Prompts Only (hierarchical structuring)
    4. Combined (threshold=0.65 + weighted prompts)
    """
    
    def __init__(self, api_key: str, output_dir: Path):
        self.api_key = api_key
        self.output_dir = output_dir
        self.output_dir.mkdir(exist_ok=True)
        self.experiment_configs = {
            'baseline': {
                'use_adaptive_threshold': False,
                'use_weighted_prompts': False,
                'description': 'Original HK-RAG (threshold=0.5, no weighting)'
            },
            'adaptive_threshold_only': {
                'use_adaptive_threshold': True,
                'use_weighted_prompts': False,
                'description': 'Adaptive threshold (0.65 for hybrid, 0.5 for cross-patient-only)'
            },
            'weighted_prompts_only': {
                'use_adaptive_threshold': False,
                'use_weighted_prompts': True,
                'description': 'Weighted prompts (hierarchical patient history > cross-patient)'
            },
            'combined': {
                'use_adaptive_threshold': True,
                'use_weighted_prompts': True,
                'description': 'Combined (adaptive threshold + weighted prompts)'
            }
        }
    
    async def run_experiment(self, 
                            experiment_name: str,
                            patient_ids: List[str],
                            transcript_dir: Path,
                            patient_profile_csv: Path,
                            prevreports_dir: Path,
                            visiting_forms_dir: Path) -> pd.DataFrame:
        """
        Run single experiment configuration
        
        Returns:
            DataFrame with results for each patient
        """
        
        config = self.experiment_configs[experiment_name]
        print(f"\n{'='*80}")
        print(f"🧪 Running Experiment: {experiment_name}")
        print(f"📝 Description: {config['description']}")
        print(f"{'='*80}\n")
        
        # Initialize RAG system with experiment config
        rag_system = SimplifiedCompleteCrossPatientRAG(
            api_key=self.api_key,
            use_adaptive_threshold=config['use_adaptive_threshold'],
            use_weighted_prompts=config['use_weighted_prompts'],
            enable_diagnostics=True,
            diagnostics_file=self.output_dir / f"{experiment_name}_diagnostics.json"
        )
        
        # Load all patient data
        all_patient_data = await load_all_patient_data()
        await rag_system.initialize_cross_patient_knowledge(all_patient_data)
        
        # Run for each patient
        results = []
        for patient_id in patient_ids:
            try:
                print(f"\n📊 Processing {patient_id}...")
                
                # Load patient-specific data
                patient_history = load_patient_history(patient_id, prevreports_dir)
                
                # Find transcripts
                patient_transcript_dir = transcript_dir / patient_id
                if not patient_transcript_dir.exists():
                    print(f"⚠️ No transcripts found for {patient_id}")
                    continue
                
                transcript_files = sorted(patient_transcript_dir.glob("*.txt"))
                
                for transcript_file in transcript_files:
                    transcript_text = read_transcript_text(transcript_file)
                    
                    # Find corresponding form
                    form_files = list(visiting_forms_dir.glob(f"*{patient_id}*.xlsx"))
                    if not form_files:
                        print(f"⚠️ No form found for {patient_id}")
                        continue
                    
                    form_path = form_files[0]
                    sheets = load_form_sheets(form_path)
                    if not sheets:
                        continue
                    
                    # Extract categories from first sheet
                    first_sheet = next(iter(sheets.values()))
                    categories = extract_category_names(first_sheet)
                    
                    # Generate summary
                    summary, strategy_data = await rag_system.generate_cross_patient_enhanced_summary(
                        transcript=transcript_text,
                        patient_id=patient_id,
                        categories=categories,
                        patient_history=patient_history
                    )
                    
                    # Store result
                    result = {
                        'experiment': experiment_name,
                        'patient_id': patient_id,
                        'transcript_file': transcript_file.name,
                        'strategy_used': strategy_data['strategy_used'],
                        'relevance_threshold': strategy_data['relevance_threshold'],
                        'adaptive_threshold_enabled': strategy_data['adaptive_threshold_enabled'],
                        'weighted_prompts_enabled': strategy_data['weighted_prompts_enabled'],
                        'cross_patient_contexts': strategy_data['cross_patient_context_items'],
                        'summary_length': len(summary),
                        'similar_patients': ','.join(strategy_data['similar_patients']),
                        'summary': summary
                    }
                    results.append(result)
                    
            except Exception as e:
                print(f"❌ Error processing {patient_id}: {e}")
                continue
        
        # Save results
        results_df = pd.DataFrame(results)
        results_file = self.output_dir / f"{experiment_name}_results.csv"
        results_df.to_csv(results_file, index=False)
        print(f"\n💾 Results saved to: {results_file}")
        
        # Save diagnostics
        if rag_system.diagnostics:
            rag_system.diagnostics.save_logs()
        
        return results_df
    
    async def run_all_experiments(self,
                                  patient_ids: List[str],
                                  transcript_dir: Path,
                                  patient_profile_csv: Path,
                                  prevreports_dir: Path,
                                  visiting_forms_dir: Path) -> Dict[str, pd.DataFrame]:
        """
        Run all experiment configurations
        
        Returns:
            Dict mapping experiment name to results DataFrame
        """
        all_results = {}
        
        for experiment_name in self.experiment_configs.keys():
            results_df = await self.run_experiment(
                experiment_name=experiment_name,
                patient_ids=patient_ids,
                transcript_dir=transcript_dir,
                patient_profile_csv=patient_profile_csv,
                prevreports_dir=prevreports_dir,
                visiting_forms_dir=visiting_forms_dir
            )
            all_results[experiment_name] = results_df
        
        # Generate comparison summary
        self.generate_comparison_report(all_results)
        
        return all_results
    
    def generate_comparison_report(self, all_results: Dict[str, pd.DataFrame]):
        """Generate summary comparison report"""
        print("\n" + "="*80)
        print("📊 EXPERIMENTAL COMPARISON SUMMARY")
        print("="*80 + "\n")
        
        comparison = []
        for experiment_name, results_df in all_results.items():
            config = self.experiment_configs[experiment_name]
            
            # Group by strategy
            strategy_counts = results_df['strategy_used'].value_counts()
            
            # Average cross-patient contexts used
            avg_contexts = results_df[results_df['strategy_used'] == 'hybrid']['cross_patient_contexts'].mean()
            
            comparison.append({
                'Experiment': experiment_name,
                'Description': config['description'],
                'Total Runs': len(results_df),
                'Hybrid Uses': strategy_counts.get('hybrid', 0),
                'Avg Cross-Patient Contexts': f"{avg_contexts:.2f}" if not pd.isna(avg_contexts) else 'N/A',
                'Avg Summary Length': f"{results_df['summary_length'].mean():.0f}"
            })
        
        comparison_df = pd.DataFrame(comparison)
        print(comparison_df.to_string(index=False))
        
        # Save comparison
        comparison_file = self.output_dir / "experimental_comparison_summary.csv"
        comparison_df.to_csv(comparison_file, index=False)
        print(f"\n💾 Comparison saved to: {comparison_file}")
        
        print("\n" + "="*80)
        print("✅ All experiments completed!")
        print("="*80)

print("✅ Experimental framework loaded!")
print("📋 Class available:")
print("   • ExperimentalFramework - Systematic comparison of HK-RAG variants")
print("🧪 Experiment configs:")
print("   1. baseline: Original (threshold=0.5, no weighting)")
print("   2. adaptive_threshold_only: Adaptive threshold (0.65/0.5)")
print("   3. weighted_prompts_only: Hierarchical prompts")
print("   4. combined: Adaptive threshold + weighted prompts")
print("\n💡 Usage:")
print("   framework = ExperimentalFramework(api_key=OPENAI_API_KEY, output_dir=Path('experiments'))")
print("   results = await framework.run_all_experiments(...)")